# Crawl corpus ViBioMIR trên Kaggle — tự upload Hugging Face

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung `priority_probe.py` với notebook 97 hostname: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).

**Bản tự upload:** `crawl_corpus_kaggle_hf_upload.ipynb`.
Mục 1–9 chạy quy trình crawl/xuất; mục 10 tự upload output đã hoàn tất
lên [AncientWings_R2AI_STAGE3_ds](https://huggingface.co/datasets/AncientWings/AncientWings_R2AI_STAGE3_ds).
Trước Run All: mở **Add-ons > Secrets**, tạo secret **HF_TOKEN** với token Hugging Face
có quyền ghi vào dataset này và bật secret cho notebook. Token không nằm trong code/output.


## 1. Cấu hình — chỉnh ID tại đây

Để cứu run đã crawl xong nhưng audit/export lỗi, đặt POSTPROCESS_ONLY = True,
chỉnh đúng khoảng ID và RESUME_RUN_PATH tới thư mục run đầy đủ trong Kaggle Input
(hoặc giữ None nếu run vẫn ở session hiện tại). Chọn SOURCE_PARQUET nếu nguồn nằm trong Input.
Chạy các ô từ trên xuống: mục 6 tự bỏ qua crawl; mục 7 chạy lại audit/export.
Snapshot crawler/runner/probe, cấu hình crawl và manifest nguồn được giữ nguyên.
ZIP diagnostics chỉ có một số mẫu, không đủ để hậu xử lý toàn run.

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus.parquet`.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
LOG_EVERY = 300          # Tiến độ mỗi 300 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

POSTPROCESS_ONLY = False  # True: chỉ audit/export run cũ, bỏ qua toàn bộ crawl
SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced; đổi khi muốn run mới

from pathlib import Path
import json
import shutil
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
DATASET_REVISION = '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7'
SOURCE_SHA256 = 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971'
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/links_corpus.parquet'

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if type(POSTPROCESS_ONLY) is not bool:
    raise ValueError('POSTPROCESS_ONLY phải là bool.')
if POSTPROCESS_ONLY and RECOVERY_RUN_PATH:
    raise ValueError('Hậu xử lý dùng RESUME_RUN_PATH hoặc run hiện tại, không dùng RECOVERY_RUN_PATH.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
tag = RUN_TAG or f'ids_{ID_FROM}_{ID_TO}_synced' + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
SOURCE_PATH = WORKDIR / 'data' / 'links_corpus.parquet'
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if not POSTPROCESS_ONLY and (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
if POSTPROCESS_ONLY and not all((RUN_DIR / name).exists() for name in (
        'manifest.json', 'sample.jsonl', 'records', 'crawler_snapshot.py')):
    raise ValueError('Hậu xử lý cần thư mục run đầy đủ; ZIP diagnostics không chứa toàn bộ checkpoint/raw.')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld']
if BROWSER_TEST_MODE != 'off' and not POSTPROCESS_ONLY:
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Snapshot crawler/runner/probe đang có được giữ nguyên để resume đúng phiên bản.
Auditor, exporter và diagnostics được cập nhật từ notebook; không thay cấu hình/hash crawl.


In [ ]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pO6TW{RP6@J&Rn8+<irn}T7j?qe58IJ8BM&iV9oS=ZRU`FKZ?nJzKNJ`pB0$SvuPX!vJKoJxH;siyI21Qe}Pu4>p6y$%Xzog%p8FINd$th~U(k^*!bMEJiR##Ww84O~X3MSK143#X$Yhi|uX~`nVhiO)nGAtNRBm8Go8uNV-$0A}%97=J-ijfc{8)S-c7Ak%eGgYPj>gwu>O!7>XjH_YJRUuXeDoa?H0XZyXmKMwc+|4TRQ#JhXW0}{VN4yxta=-anCNblM{vC)S<hiI{eN<$r;gIu^aBAw0@U;d~I7OB;-x}m%T+|<{REAk3B3|+pLoYrC)0O}=WIpCfWk)^^_q2ivAXol$Wo0E21CZvCOM-Y?D-8dXC@Ym}A^kYxQK4V@1QwLyxO7Ddt}-3=oT?mjf9ZIhr4(jK&co6rsA*yjbNr7onYt~J{WMbvkLAb0b>4pcuG3)}&oA;=mTp6|7;%@trwLkR`*^$g>MaA{kHm2#hoS&>eKvj+Xakf=2zNh=#vN9RZwW?13&eViUi;>2bH)yHl*nTq3dhsna=_B8tg-x}R5Euhsg`_iXQ=?LNb+(VkQ7lCRtX%{(U7GY&!Lc<K3S1yrbE}V@H(29h8D-6$V=vGP(f$%ACbGN`>?L+u=jaf3H^DgL{+LCobShCMlPeVgu_Y}TSF>nrVb1Ljv8uc!xbVHA<j<`(hG=G5O^(2pGQ%^EsE=OyIEDj%t}11q!Ll@iK@hv@Fr}#1$>0uW1Nc~BEJLe4|o-qy^ZztEAfl0QX!#^M4b1W_l1%J30s@z2saZAO2k>nOT5rQgw&D65e{C6qQKFt<w3ZBE-Wkki<lLzd5yE-Ek&?PPknqhFA|>00h|(Vb*K^G8guH$J_XvuxyapXkb)<9EHpsucqRh($E5&EkG<4uDDPRwz(j-7ZyBG<LbUwCb>7vWI)GRri-ecq$g)3@<%lg|p5-E)k9<lba96>T2}^^B)9$rW2xV<gXI`7XEgBt}p45TVuk{dv-HF<BWaRWUp3+E~_L)9CL74B}Rp-4|A)k<SYr!7eE9`m2agOw^W4~s##qgn0@dBnA#duCY8F9+m)}w&ZGK%rOeq-Cl<U;}Xw4|)fT4yjh4CH5$oYz2TL$yRP)^XJSVvEo}I?X=Z{D4j*QNrAqC;JiKve4SQ+>Q0ko9tyqzmRwKowiuh;`vpMq>G$NG$)ebl=giq8PE9Y)Vcf93>Nt7bgVA&TdRD;=2KgyD80EY1$Ed!#yE7*+jGgzNQ5%Q&sajf+6_Ff@A+jG<YWEVYrCSM)%ab5LKeT$cQoD^;ng@V5S&pa3SDOODQD7^J+wl3P!VOMJyXF=c;$wx0;496SWraqFwXW}=VhC7Y_y?dS3JHZ2rP}WS}7K(buLl`>Bd6N>*}h5Kd#`FDK!wWD0LiBV~1K)pv!4EAau9c5tp`yi(?j!M0k*+vncDJ4mr{`X{Fx|SgIrR^u!!s_E;0q2%X?a5n2zV5UQI{$3!o9@D#^K0~!=1FROyynXsODjxcftF&`4dUjM8eC=T^Ij+R2Vtd#n}KtJvKI@b!D>Fwhe4VuB~6F^VL%2s3Ev(>41wQ|?;vR)5;%W<hVsoc?$kt<p->KQ1aW;4SA86HZM&vZyl7(*eCa3XP?6wAm{Zw9kazpY!eBv&ZTwTFXbgV>)blv8AEfhB6SUI}rrT9+K_F^=;Bnr$_Uy0L_QRasExcK4m1Kf3pNcTsYMj;K#C%D_Qmjz%e{>#n2ojV-=&xYQ4JOn(xP%Mqc){jWEN3rQ!4^J3MJv1fg`@ADibhvtSF$y<oj4!=FCr9K36TvDUDJpWTVTss(D{0n}fX|W$Z{qg(D^RGj8@12L+>{ukN7-xqUUut(6UY>s%`rf<_L|ll);^XkrBnF|wUpFDTh1S*3iR2;FOJQk8%C3lHnW=7$sZvL&D~wrSl%lYvaGH6gzmO}0R@qPEiXLC(|2xMp7HD95%`j$tNoSP~p?k{(ZJ{+2oA7n!vEsSj<QV2`OXwMIvF6v1K{$ee9kM=^1^w(QT!Tb$F4UojG@_9byv-@0H}izdJ{gl1IJW_#TxLE|Sk`0~`^!?7w}OE}FsR~q+!E1S#@;yJaz;oEGSp48f!tf~Un7*&tnPGR9Bd^T&01pqG37(o@z>Tg)H2J@TPxwi>uJhlcoju7^#|JTx$(b|5(CGxyBBD_CTPNdn;@8iy-owXEO|JhQ7=GEHjcGdX$}=}FcMH{!E_<G3P-MTKJeW<8k}NOJ-&T<92dv8J@@w3&rbJl-{^aYkBY`q|G^c^Z9}L%$74He<JEQTSIbqn-0WBQm?oej&yLGQkcBER6S1>g?>E?kG!jIFwqmB^hdPC#>TBj(N}x!t9hQniOu0jm3N@L2%aO3rMcn<RjC2}^+oqiOhS>_7(x-A93;G}jiYNLLwsiXSqVhML%Kb=XxhY-C54^O(e304<R@TF%mbskuKE2b@?$-b==C-`HTwUqjH03S4_$M|Bv`~Z;uU$TWhR*&Wm)q<>UY>sf#VbCkZDfXw)lE&i(vi567GaXsr<$pj$3K)IvUD~mAJF1573KZoJkFF*r%91@jeY1-X+G|}^uZpSM-EPR^n=~g9qbf$?RL&a$Fvp}dvG=wNa!O|LBiERl?U*t17}gsH`_5&W9IOXR$a)cSMh>G#A}9*dD}s|-7@49BJ^iRqymUjuVzq+`aJZ-4j2^Gl-ler*d1uU3Uxiv!v}pG=&uv9W*DgwC}XVW3*!oO!1gAiLydqLg6cQ*JcU=*#}dedZZJ5s`z!#`Dy=i?o4U6}ZZdg<dRAnSStRC6R_8%IsYDVjfH|l)Pqy9}76G-hrn&<Or*lJNirLzY@G=7})#w>@y&!`Ywd+~Y$$m1Mc1bO!nJlQb9k%YVUABHbn<B(|2K1aeRhcC^)50)oNWtw}i*2nR8g=Pj0SicGhcIj)nRRX7nWsLguv)QoA!z%~x*0a>B>=C`&IZ;4?E=6*!X1h!Gp=6NMOBg)t_7k|p{NwLoJA;Q?{$v+*TF%lvh+5+$tJy72QFY}f~ejiw)i<VJGc0%MI1f$3!Td+GrQIE683fCn8diN=H~sYvQMv_eZb~GjC*ER##FNrsCO`0>$q7d^sq%YHvVJ%%e}dYMqLNBTT|;`WK7Mz&|=bNyJH$G3**@YWUi`utXbh`cr3iFF58z88WzbATdSgd7Xe)-x(B;{n^DR#7Lz50`AR9LnX9+vNA%8uVu<vmpyU!DLn3Ie9csr+F|`Nl0k&+Lt(Vkp&=z>^-694F=CY8-L0TpI*yv5=1xwIqOQ^{cs7c4P<D9jfp)ud2-!sbmzc{Ma|58UK1(9)_otKb^=yF3}0qySSo5Pp`e)UkU?(QU_#C0VNPonItY7%&=O|*N5f(#`|<yEn>W}eL?3C9+4n*L7)7~{4!Zso^xj|lO)mE1<G4`sTwc{4xWCM$GDLJmg&^KYF+;4S0gWFJu@)Yb+eX0eRejW7(ib6mLKTw5EPK(%D(Gjb0uNTfQ6BxY4U-ojmNv`s&{;E5-t=;Cs<O44F$W1txRZx4B{ML2^!PmV^=UAMp_Dz@;wd!%^2ouPCM;_OI&+v4f?XoR+Q+Z^d;`$%U#6|xhs$kf)2U)_4;Y{zt>^ST*r>^>UF%jf^D+I<+}jFlH(N(>Q~=fA6(q_VRH_?>+B;pOx1DwYn-uaUg?E0(STN$2LK&+cBHe_n#j#WUbzVWB&=^78yU>blX`U3`_YeIkxX>ab8l=cCK#UmI{^OrQPmhl{Vqu=|hS^ZNAx<SbDP@C7d}pZ_Hx^lvyMp{2<e65ErDXX%Iy<;639XD#1#`m_7a#^on@0=IZ{+c3GP_U(Oj4M471H$sEe9qSakJGM~lHv2F3pWpnEodDaNN+f3QKt`|#P@#dc8HnEHyF1v1Fum*OIpjVLIGFo)1jQ@6q`h=lU7ml2&B#QJ9jhbs&7URAu1`Ez>J_k2!Z2!g=UTE!F8sAfCVjUNlh7b+&ATncU2Pp~W(*(1{6vyjrt_<pdD?3@GySwGU7h5HrjrI$$_q2)wN(-5^-OVXh99D|GzBG*PCz%<n-ka1?4MW|H`H?D_GGn-<@?%ukKbYkqs#L@=d~fQ`)v29XKN>|X7<cqZHJrFWwi7=tV5#N^tA%--8b%Zv5K8;bcp+XtM4IT{|;2RJuc=dvfAf4)A_=r3wC0iZ11`@vJYvmge8a8fl8yBHTd+ANV3;zh<$`5<~$(c=NnHRzSUCFGCtEiXa8)Te4U+iJJqPWiDK_;VWQf$K|yjg$-$%yuppbZ4RiP6S%@5S@onC=MSAh&m>tur6o`LIS+0<RQ%;KnG9nP3;gW$Er!3KFtUwI07+YN$Ywg$zfB6=_(fW+YZqL6NTl1wrku0k45~}+Le(AP1NBqNS3*VWO^OSPy7wnY1OMKerJ|*B&*6ntg{<(!Gee2`(m^c8y3A(v6fSl+i9-mCP#?y`&>{{1nX8!SY6?f~tY(j320z+qy59eCl5A_j0(Q15fZ<~EM6Rq`o`uY2?$!dp9`bS=WPKi7n@oF1jIOT8Cc6KM&Y>KX7Y_!&&&J}mwyZ7Kp@c90tcOF0SlW1A4NTTJ<3jQSJ1!e^O7ajD+H|R`*AjQTf2y{ao1oVd`L14#uRBG&6B8cZx>PQE22qNy1ET#V>W#zxxsz);',
 'corpus_runner_snapshot.py': 'c-o~K+iu*(^?ko$syvYEtSC9M-Bb=A6kAD*IF>8R35xB4p~%_YiMS-gAtkG_{-J#+`r05U0<;KP^wECkgM<8s{-o#3g`DAKB_#^r-JQ#sGv|JpY;0^CRw*f$l<;L((3r&8Dr70ii|m*Znl4yMISJF4<kfM)_>w$%yqOo2)8dq+3le2TUU3pFX>^ijEG@aWv9U2?t2`@85*CX*EI1v_i);mJlY~Yk%Ti9%f=5}Emb93Vn9jp0DPtCudUM22^;h|kuw$<(SyGQ?`r9`=OZE3vST5yxIeddX`mE<Tp}zBTJ{o<wfB53b!876$M_yyIsM3@co2NU@=<)t1doQ0Jy@1(^5rIFaf;$J{f~KXjOPnvVKd~eUx9)q}#Qln;adyVZvm<hM+uMFfVBo>Mhve0Rdt<Vf=L!9a9)Hftt@}H_^>!W*_w&z=zIZwz2|J<WQy^V7CXbeYGu^to0|4+p@*<pv1yfs`i9{l%Nq7$D-}NTM3CohM@{+^v9r#(Y70oKdZu-GC%-_53&0zSriWd||?|ulQ5)3df2UY~Y<xK4VBuqHflVUK)a=|K?ywF3f1J!7h0o4FlbN=1?>Y$!Ky{n*MY;4&<jGMW#I1g&Xe`n6kLzYwp4azc++b$K_#|5weShUJwiX6g)G!C*fId>+_w8h3}SxR9p60{GFSsE7Sdf0}VWEQ$WTLb1<$jX3IAVkd7^_xk9Xdg^Qm!r`Lgq?`%^Kb>S!QH3`&k|Gv06yLo2d~1E%_%QI{?x1|P(V?4VHw#Xj$Y&8pfeWx*t}jphx-T1kT3Omd7N0CIAih~zdmKE<^r$+AKZW7%yw(`6(Qr4d>JMc-7ku)aGe)oDM84f_+h%Bv8PTyW95?cALnH`O<kHsSqz55cdBx}`5|Bta>7fX#;Q3|kp-?k#m(a4+)cm~jmbPK2!11Aq~t!lGLCrgljZJsrs<Htc^AdhVX-q4`wFp10>2dZ$0cfDyza`6t2|**SW>ketRUBS24Ss=8m`IS0wkh{E$&VLktD_+a=khgO%+^M5>BqbFB=zfmAu2HAmxU>UYeC+pGRLYUUJvGfJ6z!SfPlB^okNfs3pgh3q4Rf7mplIh&fUfz(+wh*ThTTWhxfT))8{<JgPkf&~WSJ;Zw3=9Bg*91UrMsT-s!n=PRlzAJieN@K%bx7F`SaZ6%dxm?4f@#MLx+Y9N{}8Vuutro)6*I=}urZ5aBG3!{i~K2g4R0veWangT>|Nl#7(04V~K<tof3`sA_AZ}g|Ggka8iRUU&dy9yXD!?NPOeX=jNNzTh5N6(G>tb0vH1y22z0OXhyD79=^mU-`5W5QW~5=Eg3PQd%x5Snr8L^0g6@|sg5%>ooof4<R{maRS=ttCgX#grJ60a_Fk8Pv}<$@D6dV%E}YDQ3u~qJp(wQ-Ro`sju7;*xnltoysZLSXQmTZzsud7@eTwhYh|IXAQbFtL1uJ6H*Eu@gZ2#Ae|wqDiS&LRat~lNn;RP@XaOo{GdF~sqd69wY4l)$wTyD5TTa7^hZP2&|J5rM~?#h=?o=Jgea{wGQ=8W3m{tniV!ArG)9dN?}cD6=debF9)1DgbnMy!ZZ82$x5wmtVza+z&4lbI;%lFAR<M#<@rO2GgRV-uk70{0fhs#xPz=u3nRV@@^DM8DP^J=V0wkVQMMMLDgRzhblco+Bwi2RNNIkeCqXiIvDqjbwR#aRlRY-voe)HN@wUY*(flIu&loWyGU{i6Oh<0XHA_Nf+Xn<IV(Tj>anh>Tf{?~cAK5t{h5^}+=C2FWOEUjpBQg((J(Fr}@HA4gVEkiT-f*86M6ch=-0HXu=kQpX^v)m63)NaOU#!TC0>w)ua6(+1aHvs67c@i!pL}A(V7wHmAm){S#&qrC?2B4|aY+$yHH&c-objv_H<I4_%va@L$+Dzo;V#cQC48t_%%e{l(@ymlpj#wtd(q=I19PmYb!t6|1UPR1I6_saWQ%t4~Kn-C*0DPC)21xm)>I%`(W<!a^42_QohkIcTnFE-$?97tc>o09zRjS#vecr4CUmVKO5`i@PNw_+WLsIM#ceCid6j(VEeU))2>zNlU0~^O=I&-1d?W#fx>O%+OZih&E@piu}(77&v-~memsAB*SOl}t767qpfk|2l*8XHxLaf@yUMKE`oIyI5&TuMSWB(?d&H5zNN4A;vOJuvwzt`oI}K);ygiqyPXtss9#`y^rn;rVy191nB-48Jy0$ZZ@)8GyvJwSjCJi`v&z(#MVyEI7dNZqb!U-%`VxbOR90Q0tTy0!sy08ONC&14;e=wZ}^NQD-%BBlfkqcZ%s8un&o*<W4ezE7~<2Rgy6rL1?Tc=cKQkXhvQaepD3{WM~@9*(-&C24G+y-6x|-EEI6a6?H}I*ZZeic&OTy)c&gDp@zIt`^X#ms9)@tW_3W4hAW89ka8#;(<U4aP62HtpdOTQoymaQ=|9&wS!U@XI1RHPuChpNnijU;3afvO&7X~p+LbC0`A{nMUO;syT|7B(Q+kHwBHwWucOrxf({?!6dkp6AD>2|o023)dcwg>IS!!>U;<<$I3wHHS;tgD+Rx_<-J1nymi-P0x5)?~Hw<NASA1$)bIv3(dETB<Wmd`dhTR6)4C8AP~BK{tE^zOTeoY7+r$CBvXe^ZiNz5N>_Wl^1z=<3anIr6nFJI>Glu1Lw={ZljqGE!c>{VSCCu|^YJz5O917&jwc00+25l+J*CXIF2&Uy=Cg%@3*92YJ5n)1Tjc4^Xe({zxurgL=7jVcObDJnA7iefRf_{PdTf|8w>B`-og9V=y(1?d<ZQx=}N&Oc<md?OfD3s;}L^g;OB)yL$7Fk|bgXE`%rK;2B5)lgp5ildCuXE6KMNT#pn;m<81I>5BaR<=)dLM}G(o_n#je9(k*{_tws9@>q6LUX!DSmWpgJh`%&rkf7kUVf3jOIaIntUiX0A+}tGM-(Bki);8LSPL_yHj<$u3UI<yyanF3?HMzjmjYGX8%~03(8nZodhnyC5+qhcg+?6X;urb3D636;|$Royl(RTpOzuqvV_<a;(B<Z6i(1r;ve0Fs7e4@y;x_a{+&<lC^LfILtlcB`^OHz54eBCuEUrRR4M%U<`^VR;x!T#al!Ql(wE$;ba>UiHk(NYwlc}okm2I@kvlt^6&uG0R57E-G@TTIBE2|3G(6IyWp-UJHruL3YozHCgWC;<G2NMzM$k}U#?!~V|pwn~T*abL8!@D9pziW4v#x^qXjv*5fta{RJx7l@xzyMHnh83$%^PBx!AlTJHj&I(APX&5b)?+qXj(ajm*HlU*8tSzd)Oi$A6EG6c_Uem_FN{y?LYc;H9W30s@ipN=&tUJWsNLCe;aiOz`+w@sx-rC>q8EW+y*Hbwfb{{>)=PU)W*VqH_nFykd-{d2oY`1c;ZnYc}?aLZz2~~DuGqCOKbdX-1=2Js4EWJ0l^sFQwk-O_i+TKy)7uIOuA#?%SrO<0SI>UW0d5;*1bPykjARc6#Lka|8rE1k;;6+QQNGvecR2nqub{It)SNF1Zdm3UvAif7ZA&H|_uiLIi)`1baz7B{ifYu-`n+BR*3-<syj85F9E!gOFVD*et2Ro2+G7dIn)7aC~FKRUoo9dUbP4s;<dv-hb?TX1G^)vvqdP|ui)h&ETpa~1o0)~zpzI+xO?R_eQ`f&fz0dN}pDhxi~JNnEsLNuBfg*Jhyby=grCOU+?blYXhvA0OFW7oN(c8sr4Kj$f3gwZ*g@ZFnYOv;Q@`JxCLQ#O#q7^I!lbnWhVxrb4y(p95(J?pGIJoxzFDA@aSA6_B!Z}B*EF(J3#0)uvUJl&pI<rLf3*tC(d$|)9!D-nmyt0d6zeyh}|w7cWJbTv4LS>boPZUd~Yfwa2a17iFhL{8v=ky&*PEdndlECh+?GE4;^LgEBktwdsnoGYjZ*H)qw5_>iYLe59#DS61iiiT;HLT5O^tLFedoMR#prosXiMI|0lfN?C(D3uRwR41c(GSfV-iA2e#Iro~fuR!bOEzx?P-AWZ~P@UiL?SbnGf4vc&Zgee6e~f{d8-^EMb7FUg;K9*6VT)zilC(y+{q_l_+z;lRoj>a#NDb3|9&!tCW)RWA(26jO+G$hJ=XKXt+PhlcItUCXpbjk6Cw3R$`B06Cd`tu(x%TAsOcM-rFWW<Oi`b-n;rfnHJr+-T8dAs?K0Sx%I9Pq6n3-{$KNJ`09-Mx?)GV|gHT4|1);IxWQ}x4dsSXC@*to1xna!91di~92Z(Vw+!y;{9v1%E#rd`XR0ydKYeHss`U7Jc>eS=^<JEPb_>b{uK!l>I2R(l!~s<9`Ul^jj1Zg>XHc%7Pvs-@lMZZZy*LbZd5v{Dr7QXbABVT@zA2X}hP&0Ts<xip>#ussX+;OdSrt8PAP)2_9vC*qOphs-fxCgGql)}dV&WmRqtninH$P`-)5QQT>_B3nIL`;|iNLuWk(7(A0}r12NMNI2d8J_pYhS4mx9Z{2jD3h2njf2SRBqh|t=2UB&s+4#l=8Z+m@emWR9&>rvCuiJpFS=qjR&;2iZiy}~UOe((gp?XEf0}fL91qIse#m{yPEy4PgF7#&Z@-}X;{~R;v?f|ed@;+zQ&2alq8E}q`;cwbfHwugfBz>)SfD|7@ycIy*APoGap`e*SsOJ~@(YE&t>)m-6QM4&O=Aibzrr&n!R~(~3GoygsLL++1ZbGU+Yrzh=vXm}zSV=T=hS(IaUX^<|8sNimnURB27ty%6UB1i}P`mr(xP*YPT>M9sKhC+D=2HZReQ^1^#z!RGzy$gVuuzE0539JScF9X(vf|b+NDJvkYWD3F3&?}g^0}CCW6C3VAB$J=1E|?WP;)SLdf;AzP;GIsOA7gxcsRAp81U7f%A{LI8NItUH)XX<=2$c~I5n_dYTK>0s9+ZMh__rIKlQ5_|K8f25KL}DFlp>uyBySWQ>bU}QCNFjw_E3}n=-vr;Sj74;hx+hK5!Rz_0xl_uH)8~AzTJP&8ju5F(tgI{uER++p?xgg2q0D_g%}Z^zw5@ze>RWy`vw9m@}ZL2EuuuDq0<2eI;(ZlPBxkouZPsome2bkDX+lsfb~nu8i8*fS5H>0=qgcW;->9#@vc|!{&}g5Hke<Dr69dhItT(N_L<UQdC=B#&8V*7I6iW)P2LQ9_*tsj{Xk{A&dw',
 'crawler_snapshot.py': 'c-qZ<ZIc{Fnc(;N6@?tW>Xv(IdPb7Z9=icqHlVO9du5rsjZst7bXQN8r@O0NRXs0^<^xNx3mm?HgC*c@H`c~S>;^2D-Q~E&;*Jk;_<tw@+2+&zg?rvIE3+!Ar$?B(b-<{)D)Y&=C!c)Y^TLG-Ure4$(wDB^@&;LSI9X3;lVfk5OwuCoZl>NFi#R*>Cedy@$vfUO9n!x=7A3P}w&$gpm&}Io5j+PME?ig}W$Dxl!_lHxWN{dJ$#kA(g%{0cX%Q7kI?LDA)VpkN9%XsVbq48V5)Y_^s&p+~%!)Ycc*A%UEhfb<85FAUY_Q0(cvb}5lCE=moW;@bl{B5aa1;*~MGDoUJRGFc`6Mplp{`%VN5v%BRrTb}G@3<wamJ;FQ4tr(G*)HQZx}xO{YpBMuE*0TnFNa>ndolNz_5H+q@k*(2EO-|WUijaQ9g!lsGs}~ya=Xo5kbXBf0;+aAq+HEFJ8~nnR=c^#hCjthbLoLuiyuH&Z5~co$6n@JNX#9u79P2{kYJ-aMAK28R%y7WBpjrT*J!i7senLvt$6sLPMaQEwTxYi)Kc358l0=CbJGc<nu{VSVgmRH!Y}qrYhXRSD#BJ@hkLN<I?;%%ChtjmKtWD@-o21UlcKY98^C7d_eJnKE6>-4!p~Ya->-_N+uCtnhd6x3{{u*;vyVPhHGnIeBsus*I&Nr^*sOJNq_CNtN5fg*R42@qiis~hI=f|HoC#4zjo{8=U%=YUj6(FH*YT~>#wb?!4B{i#h}&RUi09;EQYSkG(rZm^suFH5)g6F4w5__r5WHwt1Z=s$sP=;g$rUs5flvaakO>iNiGnK<0Gk{>ca%YA(m|+jtPjvq!<&D@yj5c$Fr87?fPw;&=D7Mv_-(Afl<fvxozLoZd9bxWDxEi7jaIlbUb4W_yXpx?1FxaD~kkTx<5>^7XQe5x3dN8U!Yfc5vKd}ldBZ*bPj;1CXOncMbnr-2KeMX;f?&@M5BCw%bpAYW~+S~6w|q1f(Z_@1a?06nEG0N%HsJX8pP699w8UU{Vd~4cVCy`IR=mAg5hF1&s$Iq#xl#1)JFLrNqV1)COM1`N%E}dZMB1VhP$xkFN)FnQ+``DIZ2~o*`kdP)BqO9TgJ-bxI-k7EdmKs(5nK)iWBK2Y^}U`489-WzHb?0$y_68!?gQ%X1)iw?BPknaGORoP62O#c-Z4<DdIw(0xU!;3Lt&JxJh~#XByjPxM?QISK=0qRRG~5;D$lcg0h@{^K;ky4wn&-YLWdr8FaowL$KbecP8{K4)VorEAw~m6yBXhw;Of)Phde$ptsFI>C$&*Czm>>3{W@5vwIZeMV8E4ZJNF#oIZpT@MKU2+HF-v#y^K%!d7X);jGp#T@-4E;oAUyb61j)*TRn*o)70{GJ;(j;EKXnX@lj8Tv5NH1nB-3qsbzE0SIr_@?TyQK$g8|f~a+jq``|k1?U+V;Q|i|n$*EKP2or6&Ei9EeH|JNWb*QKkqu(2fBZ@a!3g*C8^J3~<?-Zbncrau^HGow#&Cv*d>|lh<ld)x-j1IPaXtZS{66*qSQwxWL2*2fb0Dx`0xSbC3#}@IQ4&vvEv((9!b@l2g`+vJJuw_749!c1z%30uTo&N3pmAi47Sma-hMWv@9u#lMXjhM^ql)&Z@BKFaqeco9gV|y-0d5EwD;&{$5YRJ_VmRbygQ5jx+5s?3^J738n1l(?b9}dK2rnghF1+Ay0k}1QV@WN{_2+fJsU_^X@O3SIJxVg5#SI@H^Ip4k!;2QhI0Yhk>?OIEFGizekN|xnw4P@v9465GGu{DgNIVj~5e)L!+l}En{D_Oe7%4Wl232+=c!RqE77i$H7SanKPUY;;nj3@0YPb|oi3$f?wAF<bPGPbp37%MGd6{v<m;~kq*r*xoe1Oowg>5Trf?YgCaIlZRG%kvH{+aEKjh!!T(8ezbnfYIB`bYlt8I4>`In3C$cLKFPm7U7etN(!i1HK(eSvFB*U;~=E)ZW?ZYf5O=;C0#Iihb{ip0`;YWWJcT%tZ_iuye5a+upO@X1Cj|55g>7hYc`!BLjK$K27jzkYNJLGSJL2{M7Rkp%CC<6@`<Jc`v-Nh$icJC;%mzqyrLCn6JrV+!k4~yP$W716VHSuOlEucp5QYy|DoK1|}9|lmJ=8{rEVCZJEK=mfp8JUYA=v94FWh7#CHuCQOgYMm||N_#aiNVH&;FQ~7Ll-H4*B+MfE)hN+h)gRR~ZX(G5q{|MoY<DojFfH^-%VU=-8O6jeI`$PzL9M9^sVA<|$S7&4Z3|L2POOUBrOSWdm-nX09$I3Q4|1QD7H1Cl)NTYPXt4UEstx}U@n&3fe&3bj2ksKp9Tsf*xUp4$3L~~%ohpiK&|J!=1h^%dUM238BfxhCysXZc)@*NmKqCO4<$SEQK{C^J)A7k|GdYzF}YwWS=^$wNm1N`&!o3V+vcF&U?*UI>Cz?ny`QBS?b{gS1m`RsVGVSAHkx;u=#Y}?D^1ZC(&lod$1Qxp+^^N)%mBb<l+B*T!t1A^QH@e3fi3Z!%OU9(P93r{q9jfZWET2OD6k+fHVX-GzJ#-%`s*4LBaIzz*KE<XWMD?S=b7I|_Iw>9%H@^rg+0%1#KLwP)5W~7O~fTI?+B%F{fVU9lC^E(U?2CiN{>+@u_`_?uqo2jnW$;FGAxwOQ{XX+R}E!RqQaQS)}n}wUA*=?Y7t^@UX;>I$-6=F+I3`nO$^ihNqzHNP-&Y%+w{xmR7U3D&;@YuG!ZiSpPp+2=M1dBN`GA%_NVRSM#ePb@TLv6*RuPW)Xm?n%n10lA3f~&?b`IIWfQdO9+AY_({D?yzT-FF|S29^&fw^CmL!#NFk#&9D{X?cO|ga?b)l+V$Te2$Ln&t)5$#RSrTjdh_a9n0{H>!c4`dMiu@eB%cL5HlgjpoDfwlRN3CDXNsAWny5bP19uyMb<<OU~d8Pgi(QX4HY|x{xDy8jBos_*FV3=7PmpidbFX`Lb$U)O*u;Tu*4wE=8GKP(FujGNGa<ZBjojhPr${<7KTlor;~#i4!w?EP|?0@8#*2ll1yMY4ZufZOR!XGgZNI8Lg$C)$H-jq%xMyAc2LnyfF^{BI!n}6(alg3G5VIyvERUHz%R{p&?}}4f(3%e)&4uj>e>p%>?}iS69HfzDwLl@*f3kn6zwl556IaENxH}_$yJwc8+tQ%h4gr*Z*f1U?9?>CCTQt|=(lQM1N#Bk0-FV%M;A_@Dih078tVm@&;l?VGJGi%%s^UP#5NtJMxf<Vw**kyRp?xdJAhPK_Xb&d^^rwx^_70DAPM=mNM&&S`3)2rr%6t-Rp2-D0abXQtBejqR7#=75)RL;Zr6LZXO%#~4AHcp?bb!s>yk^T0Z8_|tI_Bt`n5W{DQG@=WDy94F)FH2#ovM&eEPD|tnK8gL{Ew!=zKGq6D;q$nXRUo!x7+)Xq-Fdjt9_V&?9BhfLrhArpbx|>IBq!`2F(zD478r;M2T5Z#>s><N^4ziXc$|WdIeemmN)_y%0q=s02&|)vT8S8Uu?91L4iT!cuXKb83|>A%W~HAc4$}k@nyscLeHLBW5ZPqFmWSTCR}HqUNBv%hCa$51lJLI#ud`>&KKq9dO&D{!6W#t6$g?C104Exh(5y0ts8~Jk%6+F_S~eIWS*_&*we=5Slv&C0Svrkx!Zqf+i*bdMIt9Uh@=Qjo`Ey%Nnk&4JJ{Zdt{2?z4F4*AfBT_KnNphO+z@qk|GR26q<~P9|mH9cFO`RgFR3QTP!xH3#kAy!1PdlT+C6n3^Y83fTWR2mOc6$s<q;*u@5jB?7$>C?rjFZBf+@@_6NY#OC9eb>gPdp$V2@WkEc$ZSV%G0*E5*Z5N%8VxVL!)LCQxK6XeUKE_`W(1Ip{{INV(f0q1*9cMTvT7$z$Ji(*1BsxwBzp(h%XJge@jQxum_MqQ5B22>e1)YOl7GE}~bEXM3i<)bNTAi-<7;I|c5l(5Ft*#}F1<tEV8D$@lhtaec)K2P(c0PaXBkWHFxw#KX@3D*^1i2sgmb)08M`JeAb4&0*7539m6@u>u6zFi$siEbvO^6jdS4D>7k881&Q<_tvBjz@|Hky$H?fO@TI?$8$-x?-vhS}Riah`PA`xdn@k6f2F2wl0R=0AQaOAw7_i^o@H2+)Z~SwekrJ<5VLWLn9tZVO$fY$tnS=wHh$NwWw}b)?!&-jM+zxl&g5Xq}|5eA|j5pwCdgf?}t{UC!iGG&!z1d@IA0W?4Fnf_!aK5(~j|f=h#4nkYvYHw$@@avHC@C&`77KJ+>JFt_iiz;yqMVACp?dz$?;Jmt(C^8D&c3x*$mvqN{BY23M!WXg&uv1CBsy-Dpmv2jC$|3@Qgh0vc8ED_CEa38Vz6O4<J+cvaeIpHT<=2BIM-rWC_;;=cys=K9q=V9#K~zIGKlaCLwOHwntHiW}o%I_Ur(L&^jIu`wTxo_V9!4W8~?+_*?jSNzj<)q|nlHqXju$yH_mBH}n5>UmNosg`ZbVfP@`Fqfuja~in>+ZAIuKs=U3Q06CNk>4(@l2qC>ZObd*m9omKIj|3hX$={O02q9ZpRn14*mhUn$oob0jQ9udDLBwCU%u2n4`>(-R7SEk#NCKcjpiQnuEG)>dqt|uTS$T819VDLOg{Lj;S09(d9*egz!)B&DG<2SIoQGrG1DjtjRcrzGkH6qoQK|3$!wIiDs$Pfh6`0&Fu?i>^?GVim{)kJ+ioxiDnG6unTPV&!tU%`?(cN_=(!W_P14<H(nO@qiCoO`XcQ|rey55F#+H>m6{dK6Fx4Q1Iw|E9tH{fEkv-gypW_!p*j=FeiF*|-25kll$IDN6TJ)566q`Pg5QjO)KC@zS%pA5SMm4)KQa9v|75UYaP7`B*ZsDZrq2k!=$-`PfG<ahH_+rqKcp{5peOE(A9b~s0ySGl05=>L5k%B7-5X3VP9ad!y$PK<u0tM0i7O#ZI#}tT-;|LB}pa+K$oU346Gc?55Gh;gdnYNq6Dlk*hSM!V_<ps_Y=`0+buVSY;g<6k?yN%lTbZ^;Atb(iOn~U9JH3b7;frjbftfGscF65Iqp0}`rm5<eV(q<eyRx6!%2)VpfN=8OK;r@TB16m@tM#eFN4H_8q86^i=Yg(y?&I+WpQgz%c+$S{i4wX;CoPZ)eZlNMyn9;I>8{QTh*fbMdsqWBAXii1(7Hz^{A{=T<R-lkGMI22<|AO}WL#_JBl?xddaMT32Cv7lZ%#f%Qpwgd^sui}3v*d_;k)X5~6$vO?sYUhGdh*JZOHa0&SBD#Uf<0>pR=cq*3YJv-)5}b!&Fh}RsfZRSG|G+qm(Uxg7)LX(XCbsxuhclwh67cjzq~V*jiWFY`gi#X?TtnnQmbmvllFPXRmbP%Zu8<Qh1HS?Uq%6!_5M$`5k29)HKH|)QfZrn1@0)Nvc^v?5Tfs5r(~^R>nsN3J0DyFv(t4bIb*r*`GgdwQd_{H2ZeIm<T-WsMp0`yluVTk{ex)IZCn9Sj3GyPdF$zpccpu233f1bF-EGS0MgsvLRoPA>IilxPIGq&k~oixC8JV5Mf4UKyBU5=IzvOeUf$Z7-C5ctXY7G=F~Db-<mtSnc>?;pr1QoATIC2j-fhr%$>YmFXkaD|>1X@wzM}}@#$|(n*+J%|?>Mi)DOioR;izIs&9yJqY&SG8OH5pxf~y-Eom~u8$JNUv8=#zSXD5gu%|Me4JJS~XL1Dt`1EuH9lD%YRI*Z^}l&sLWK_AhHdV?zk#Zlp}Squ@$p^Yq&CW$>=`iVvbO*tl%#6Vw94sq<{@1JmIPPb1q)`c@rf}Q%xox9!M+1`Azw7h^R0NmKgyZ6NB+9192gLVqi!-Y50Y}Kl+`hq53d30<2d_@|eB(ea$C_>rjFP;W8Ty%+n-^5>(s+)NVssvc&7&ig*nMEh1K?I69LnhYLCom)^N{}o8RoW&#gPg23J8s?a6W(TB&^+IHrRG^FPiO!={}!vVsGoH%QN|hWgVR>eH?bgG#!8x_My{=H5rtE4f6djsoyvmrOWpliQxo>YT7#XZ^Oj$Z!E(<sv=1_VLVj>i$Ec@Ra1a<XQTKtpxr3HpbaB8F+tyD24-&(6w*6{3d`)RC5rNMZVAun7ULr*kqY3R3>=W4}(prhr=sdwsgwxzlaMYNS^VOURsy%et(1cnQy>0r6%cADp2g+`|N3NA;%3x=ngH3D??-Enlp+~>j-MZ{uq<=bRo>6E1r-TxR>f6wmXePI@K^cIl_B?|(Q!H6Czkv&&wzJ{C&{LCU8tQu%*L01)tmEm9Y?I$!ajG<LE8u!3``g|D+e5$_f=H_x67vUAs(s~83*R4gNm|P9V7-t|D?e^uQS9|dv4OZn;i-MP;;;~_#GRsNdR(-Zu>FV19x23s173l$c0(P8Og_qE5B+ld6LmVBhS7;^&eJeEf$ex2?0v;=Y|o0XQpQsBX+Zx1@+{GQCG4#+MJ(-g+h^d9=?y{0e@1>O=#d8opra<1QCLN7CP(=j-fgdjtbF!hTTlo1fh*l^zauaKSV+e29+_LQ09eXV%^8O6<x-einr~wND$@X5+Z_E5auAni;DgX7s=WY0=MadAz@e}BQmUzjTiKS{Po}47mbFu*Llrlhwe#Y!fH7W?vyQBGueSQDh-;U+z@c2~ZsDIx_~+l_pQqTX%Azng)aG0wD3r&egb0^JTwv{#aQaK6f0w6PGpo_;`Q9MU+q@kL^NMMbqo2L{5r5;MlOcjx6rVGh%m(i=BT^98jl9Lm5J;kP6kg~zE|&i+n!J1!1_-=1xW4Lm28vyL_EX?fMOf8PYS~^<{D8qz;ZD)U9%}tO%97gTe&O~@H#{_f0}6Pl^41`krri)#fGjz5q7paW(oEN~dK|0XDq|+`9M2o+o}K`#Ktsm|2Yyxb9No)cG~4X9xBqlQ*l}!;dOu(TUvAX5G_LQ69VAZ})Tw{ItphL<(N795{e|1NU-2}w_IdWF!T4JP1C2dslL*DHRQB4F-R}R(;g-m}`gmEzVh)Za%erjSAm2Den{&%T4Z_)~r@fMj;iGbYvmIXGL&uo1wz!ZtiO+7Y83NMMe6*-mb?-n^^(sSepN&GTipveb)}S>@0!|9A-e90-AH;YVwi;npyW3uoBSz9~dMPWbyh}#gS>NoRNx>>OUuU=6)MVLelG(mod$gC=wc!n<r@jN~`^l+HSbi`yC@t2`fS|eIl0$VF`pfPb>YQHBsm0k|VW(9L>Y5)~W>l9#_Hyq=gZ*@bf$>}3#fx4`^=%zANuW-Wz)nbZfC)s<svK-xX*U_21=t^HdVVAevx(ifkf0xh^@(M1uZf0uLk*XCu-vdxW_1OZe=dELy{l9|a4-&80bkeTf<_~DRZ!5_)YDHO7TJPHuTdxw)h#yPEG-No&{Kuuq!7bt)$>W~H>GQ$0W>^Jh$G=*rH6^>_SER{B#LT0UQg!YQ*#Qj@bF_67a-0X2d(mwSfWHKjmft6i0!afQ=Kc#dngF*m`Q1Ttu}GPQr&L*e7X&Y&pJE-Y1=6EHal_7Mr{{u;T9Y077TNT0rg8an$mJ017CCVjI|Z^dM?1OlbDOMhnBTDNn_p+jZz_OY62SQ+FbiKxkV)M2Gxd_x(DbVL#djbIE~qM*nv#PQ#U119ppU-<z31K?EIzkFvJcNLrzc4u$#v-qBjg>k{q}gB+_cqVu6`S^1qZQ)L#2!5X`UzQ6hm7LP@rGS9HoPTp^75Jx*hzM=4V=qdoPwRz>tPoWF!^(UhE&ABrK5nG3CY&WhAFh-M+F16xLlFjq#9X1OZK$sLAdD~r^!5wfS7!nz^7G#jJ6)-@u~X^%=BrCrCjRwZ*dR0T-^KBy`cPDk|x_p8`QskE#cDKU-8wIfOD=oQjO5l^1s<%rR0rlX0aU@G5^Iq^d!*fK0FE>Be!z-gjTLs7oqC336N(aD&JELE;%GtXnBA+VgRZREumt5XM=uX!IkyQ5Iu!e24Nwvm&?v)Xim<pZ1Fs8hQ^k%g<Vqi9OswaJ>*xAn^WxK=flHQG>@7*Ee{JzHidQ(axvLe^;TdL(0)n7Wk3i_x;ZwnWoy3X%PKcv{sPWCb~UsrFy#*Bw&TrK(hAU}RO8YL=p!$t2(>_`vUniXPkY95^phV$Y+&T9f5ZxONhuqytS3Q4K2Ss``z!ROs{6#Xno*{e*umv-GTn{p>2Ee$2a(0*}nrF6}M@%qaK(DnSXI>LW<)3YIa7_hh5>Viv5zraij+X^#IKtQa^MK6yT`S9F0Q{X2+AS~w(!JM9>zr9w)0%W|klg6%)y($|Gbq7=(4whq%!>FG-6)+c7w{o^gU4wEdLR(oWcjq+k0(g-Ib*87&W+V*Kp*7?NkOFHcP(acIT&t-BYBofy7P^9pa+dxw@6Vh!NhT|kIxe%_jYNiEZek^x3m#?XKb<h^@bsE@oDrr0?aAeO^ic84d{30N=3bG+f6VB`|g<|X?{~TpW%(>4{Bmg&J&?zK?NxTk;#ikLEj&Z!mC|NLNf)-4IXtI)_B4`vH=TjKdFgYL;NGPUhlm2ee-%IrOGX1?mf1kv^h1l>1%DPVwF&RtjpO)<30cau%(BBSCmhUldax}$kic=hS4pC|c7(WY)(}dJ?fS}KKFJp4S80USMU>lK90cj^P=`wCkrn%Of?ho4!04nGFbH(13aUIc~9Dk}(yOwAS{kotS?XG+PNjnRgi{f}f8!Q=RAQa=%EHyqK(5^#Op1-^cC3smS*zVLVP^Rr>ezh(xtl9}>E>k@bI0%@YWwmsn_L}WqpI~z383D2D^RiNb8mXYS0Y=L^Vo|Ji%aZQ7Er={(Mv$dr_U?LqyK(l;p02fWL~1LV30{o^v*LBlKy<QO&})uClNrtzyDEf{ynR~`5)qfX`WD(s(PvH38o^#a=i~`O_&ShU6*&cjkTdrK?x&ce2y^b>&exk-0N^&prh=n6xw>p@T-^C=GVEPYlpnlAiVOWR?-&L=sJfJ}WV4nY8Z{DYwojC*0kLtar9d}TfK4`Dr*TwUQa(~=G$bZ<-;+sJ#{jg3VI4#%MNH?Zp+voK7!QGLK-$imd-vZu_TK;Q`(K;w-N`R*!~a`%h9{eyt<w$T-^-`%)@MF-XV|uK`ASy_@R=PoMT2VoZy+3Va^ogYJLodwKR(=ee7f=YpF<K_LJK=5_!(a5X6rV($$Er%`qa;mTk#awNRPtYvnj<e5E5ZNIyfN?6J)J&Nq}iMOhV#k($U~EkyV9klCFP#tdsD=DTTjHRHPs(er@li5yi(2DFTUd&`AYa+hLp!U^_-LOmb8YVqTGu*G_2iP>NaXFIrre_!9cSF}s)~nkDbHsng*72H1#OXPt9@gdt=wiPtgV*&a;cy6i4;(2%C;lvl{M+Uj(Y`?LgRy^7+Su6Tgf354EiydgHmmv^vbV1F=8?hLxqy5L>tlw@i>pJQ_C&k}4<0K!-LOJL!-Uty@h%(LIvTJ(sG5k0`hc0LQS>!WI$<qTt5g%))|WUJY6OVa@Avbp_Lj%?u1DL)HM?(OS0y&Ko>f9t0AXRlqo>7T=S5wVh|1<{hG-PW9OH|^!2c^YPr4-JVf^g;16`+)g5QFuxf(=xdeXbP2pw5y;1Z@+e6o-<NhCHYSWlxl}VFV$wJBgIrds&l2>=TdUx&;H$2Pu7m~6viHh9weE`V6M2yMewwvgi$!a_jF=!Zgr!4AKe`B<UTw+42(C%ekCOZ-7wN{bU_Bf97LrHeZQue)phM<SqAdaaCjFI;O}GA%BG2C`5_;PpIAGy9IJ+o#^hwuqpO}yt`o|f4y4FR(9l3IyuC6UDuH9#(Zz{tw<x!ayx&+?P(5Ru*EOU7vS#qmZ(7l_M@3|13DUrVePPo=8i8`czF^xXH%Dpunb7R0E<nh=*=2ChsEg-5^@8)>@=S5Ja8}iN@f)H~o8mv?J0MB&Hc)&)G7R1-;TsWd`tX2+Fv-iMx)*}no2k@U>ZPlTQcuRB;%12PGlsXMCb^`l)-Yi;jS3m2#`SGC4-mV~>A+F~xZ^)2I~*(vf!=61;=tYxkQ}j2W(Ko;P$QdU#&~0p85l&;j4r@(+=l*LaBMfOxawYWFh2~y+Gmf%0-c`?YE)J{GCQQwBr7fkGmd7d$-j6qG=_)QEzrUI`u@DZ&lDq+Ee2`MtdUK70*>`TMMtaUfArt}_2Jk5<->3N<PTr{-v9aSf5nxTS3mfxkAC(RzG}B!?qt;idqrw8-lXpqdWevsyD2kLJEL3@cO7r4v<d&|{K?1*ZxJmv)M$*g<MnPo*PfOg<?@7Att3;p){BZOqfUu9c{ZSQuhFEi-sA6v;HNr2r_U;5cL7dL_x!eylDbG`=g>5jw<fOzb59Q623GF+kH7N#=hkKORB{;c4l#h$eYf=$SW{3qk8H0QdC509NqPnDASSIO&9aZJkYxuRY<!#JG~<z69(eeQZc8Gh!i?W+Bw&h;UnvRuaMX8U?uo`r?37NxOAEwbgo1Ru<!3GRtlgmYwoz8506}6c>l|Y+sLuwi|8Y1Mv{FaaoU_C7&}dv#ULPhtyWop|rfZd0>#BV9r64ZA&i0n&uP^14$G&$~h3&PaX@o$up=9i#6<lJYrMffYl?<&ZBvL>a!X!dl5^doa?(NZ6#)0Y-L|9hUVn$g^X8`(E_-EpM{<*D7n_Im4fJfJkvjnbu8?a;|pyW;@x(W?6E%Fu=EIs)D&YcT){QhTJJ9iG(`%kp}>i&V6*R`ZTE>m26+qCsRLMuG6n@?VP>T+MlV9<?Kq=Vf9<cs~b*}l3i3mOvO!5-*3=6DB%wwR1`cZjTNdX4sXRiAi}y<MC*Y-?|KAOAU8#|L9|jrO3-=2P8ET|NyFLEsRbFH{VPsLOE-;1-ifI-1h0+-Q8eIRjA*teO^8mz2~QIPDlGnzA6j##d@RPc<Z&z>PHBU(8*p1ZsWNPQTjFVBFLfDEeCVkmzcU970mMeJW&ySkIS~v~=^EqB5LT=IjH2l9vyWPOzb%yoHV5(-jITNiakKsRZ_>pCxBiHm+V1Z7=HsXmErXk}8mUPlgmlA4{?auq*=EcG1SN5Pqo=Mw=iSlS)JpHO{#XwAYYO&B!My-PdG(k~h^>6!i!du0S0FZS3_%c-9L|Rkdnh1$98jTbYk1wFK!{)McH3NCsm)kksrAN)s$iR{`-=UIrSrD=t}yR=hY_aQSAfsVg{~O6^FAKIgWE*7O@YOFVs37H6q!rbeqe6)IpdO1_s9JS!&?^suJ(^meq?gbYZmJr4HLgWzE1uR6!bA*1T2c_!DY-LsGrJGI<SQWx5GKre^KH>>A}M`vaZjg_<?wQU<Ltgu7K&_P@mD4&<fHiiZQMVa$91|$=-{O4@Q7A$);!Lk%0oeF-3K#Bj!{YcFMi5?`LLIxLRSfAU*z<E9@uJJ7N@I5Ibl|vaILM?o0GQ{*jB8Iro6VfbYVX1_c+o+H2^eWdrc05i0dJ1=0blLKk9SAj<riZhS9RQF@6v6%(U%Btgpt>X<xT6kGB&Z?>+tdRWg7#PFIEzU=EhH&4vqdEJmIWkZ8C}B9Cq+_39^fiUTkT5qzim+<>qZIEtXQwjiwQ_Qpj{$eB?7u_md+TQ=?%FvqE^Q&ty8y34U-R-;&Push{G{_1WOMNOo|^L`V}LAUAgqyL&FcHy2Tv$jw?D`bubu;+K`zwuUIJWK<$Wu1$X{H#kqF83co?_MQiFRhP?FtK$4*)eCG6Dr!=1+Jo$GPmo9g^J_ctF@DIFrdNdmG)vp5;j;u`N6zV(`*BLY=kc@^<pw5tTEuc>^xC}{{1Bk8#YtQLA!2sQ(=i^B30W0CxD`Lzqjv=>hQ_bz$R3$Q4-#VqrQs$j4ab<*V!>vCEY!m$3s9cO!;>}PnEgQxU>LtRX4iIVUiv0B|xp#PD%xml6eeKVHESBUdZ5{cvB)b{ex+cKbEKtm%qXNCIvOw(+oPJ}x;9(bRl^COI!eV@79AaxMNKmX_DKz;z?|<$7kKh09{h!|d)dz2X@XPoA;{IQ}|DF5ay8rzT-umF(zOs0`0Usn5V3R_a#en$)0s+x`Rvakgo=m226%P?poYt)_&6uue^qgn#l7#!BV$ofU^?`!WB^S(WOifq5Rt@gdRoyCd@@*uPIJ2guL|64Toka~Dnksm?Yk5AQQ4a6Hz>8VO$Y!CPPS6E>cOh3Y`frj$D4xc;u%hT1RBg)337vuJ!}@G+c}NH}U85d^EO(e9I+2g2N>jdMHWdDMO_RZr6ae~myPHh_M0Us*IbPgWLMX)-TiT%WKuA+plcurIrtW1;o5oJ!fxT($i;{5=quE!>rtz@Ffhizg2v`UVqyPuIeWYLMF1hrQgB0pt@@cvs%Qa9PzT*rN35$~+cznkcD`#3=dFu!~2y^y7{Pi~<{_NWifAhPC-~HCZU%d1C-@b)3{NdMs{rh`A`0!hQ{qeis{OCJ>`O)|PiQazu-bdg6yN7@C4<EhvyN6$Y^W$Ir=HY+&hmYU=F@F2-hyVG*AO79PZ@&BRdtW6z`TzRvM?d?~!*_rC@i%^2XPf-r|K^tu-~7SDAN|9_U;L0j|Mxc^e&g34{qqk${?)G@{^i#n{^B?I_WkdmYmGM>KX~uwb8xhIaPMp0?)`t9L89*kydo#SYvcPrhZp(%w_!1MAH4TY>P_LBvB<cNJY=<4w07kynGf1KmztJsHw6`FMhjBMBXz<&>jjsuG(o@wwOkN`XOh%HWI7yd8f@3*G9cQ}nczTSQz?m*pa~FYH^UkFJ%{Navr*655pKS$5o}ZWdOy#it3>9ICM>0jjU53%vIV5V8GRrI&EoUPL{Z`V+ba=YxXk5BcRA@ZMJ$`Es{$9hMMd1VVNw|)ry&Mp0L$oJR#VOSQZfL6D;*Wr@ftq}hJb6xC!o5_?oRV`oqS5^so=+FKXqpZ+Db-upQRU{zWXeQ?4OpSU9_D|V@}DX7D*j$vc()WK%|g@L-Ob7MW4~d5B`^0pV@|#D0d+f$z5b);i@Cf5pwr+yom&Eh)<E&-^D63QZ37%hj;(*_W$_9-~Rab|N7>`pZ^djl>GkiZ+`mWH~;s;Z~XFZfl)x-{`mf{kg`6w_dTEl58it_xvO5?|Hmm1)d%nW<k+LKZNENuR*!`tR@|OU1lt(_5m5imA#ldjgO?@V+<KfY$f0<=fNQbtQiebz)bJXvK*qZTVJ+erIVQgz9YlOe5HJYl*axBf?jL^o?T_F6>PK&V_3j{=L%3UHFGu5g@Ce|4P-Ozg{+>Ec^{rH;{Ge4(#K}uW4v`cphDXvtIAG)EqUzggb~F=99k26cU2B%jry`(i>-eXNthCLLQKttRP#bDy6*Z&c^q9rl)(m2qH3?BlDDt++Q=oIYNv^jQnXj~9QDzem(sWV3mRC<o04<%y5<V+=oZ4V2VhN1nGQkc2iGCoD8>cqSNN+7Gt1n!{+jFsuFJb<6{?lq!Q$43Xq62a|tVdm%nbFevOYE6+Dd1Sk@}|c6<#CuvjP4ceS(g@C-$_{3Y4apso~j6@tUg8_Yd{!k3_0-(cOo+hp3Z+qP*+WYSGjUe-=f%4tr{uP)Ps=?kH1m}dZKeXt_S#9_e)j-?i{JV>b5$nVur0bVHqudViz_pxu|iK3mPBuZmEDJ7Bt}QYGA(5zh+|$M4gO*$d!0RB1KnAL4wMM3q_U*@WsK}YPUVA;h2gJnFvL5YC*TPh(&vB5|&k_vXNMBowLyLt!~bX<b(=_Saq*-Xhn+mAl9~?h39nJJAoIczGP`@uhk*ry<MUX?+Hm*Ic<o}W!QAX@$!phZ9<@Lo5EKnk)ykBg{Do?y^xdV*6vx?pL`7b>T0%zrbU0O-nUl*v@Y4JU$9peJo{=rzW<`Uw^v_ip>DXeZ^3ol2Cch)#x=yKu~ap8Dt8$7m);Q1cdQe&gu@X|;-TukqzFjo%5fEMdfTqdH{Adx*MYUmTkYfx9_rS;-_;8R?c~?>D+2ij$%ZTc>`NpYZu8^oBi$GDA%IgAoXE$UoK|Sy=BF0r>815k7e?Z<zWva{Q`MX(S+XIrodsRn=F|NeDP|@A;tRK4z5en|JYQL!cib}_hV&fLUEkQGy26^<uRAXp^I7SekN0|oQF*=Dn)~KLUBt>Gaq*%$yRB;p9YE&CAVtl>a-8{b9>_0cZQ^hfl!__JjLWE>E|?$_irdV|d|T{(psy^wd!zyMrqPZxt&)+!){Y}U^Fs~^w2jl^ywg7Abr-%uHWzbMatkAE#5H}X8Jy<o%4_1Y%N>m0XeX-0U$*h+B!L(Pxo=bU-ZLIvupGhpc8DR?9v@g369Y$~ILTu~3`Eb&(AiS$RfFa-FQdAkw(=n<rwzveN;b_g26UAPOr4Xm@CDOsj1zTKPjR7^Qk#|6!Yr2BeB%~!?1kxA1F{TVx>nBniZ_sxAB?KqZ~=K99mMuE<7Jh405KMBxNKa$J>p&C*UC|9lCb2;acazY<2W8xvO>}$Bk}H%OW2oo33E`3#?<Ug-2^AgsxNd-4+XI}3*0UG2kv44lo*3;1>72>hCaZEGJ)C?6iSCisxOT$^0c#U7GxAsfx|uT8YwPSv@1Cx-lp3pUx7fK7mnh=0`^J^D`VCMj7cYG@`HU5p`;=5E+~CEqAPv~^2P2HV@H)#h1prEVwH)~(rY^p3`G6KM;(Dp2^MFIX-rvgA|#@$G#|F4aiH_n^dbnoCjbKW>#5#t$#b$p*r@o{K3*8LS#yoH${-3G%&o^`VU(-L(?rWkA-~C2x>lWqZkMyO;~s8G?tx+^Y$?sm`g9M6-iF349$I=_!Qr-)VXYaEdolSn_uQ0;R!U@FS3NXN;<>;neMQ5Zc7Up`29Rgc0K_lavS8(Gz`e6@X#ny><en(bX7=^w)kLa@-`?XU7Y;Ygc?$FK7dg2K1y~6U^sWfeL<!yx2P(h9w%SiVrVkn}Ci>2H)qvJkC?8wg8K;ghfT9I>2;^Kz_Z-{=@rc8o@PY=-qhsjwXG!EeDuAg5CL|*<1P)}%3D_!?D-fFY@~eCgCCuRg2mv`Np@5^NcgV<bX1@dzNYXt$;M@vSY8sL2D_mJhR}SEjPD)I82IClwT*tkcIu9a1a8hzYF3MZzm&A~`6LQJ+qbGLR`5G9Dp=mEemj`+ULG=)uOq%tIHbK4<QvqJ&)hn^&GtI6+mgIFF$NR0WrO_}sJyPUDsNtopir0ZSc1`nxLFIBoXSyRUM-xE4UeYH_Fq5M6D&KFs7~XvD#<drb#=5d-NdltA{6l<v-hyKX#11(?uZl`yz$~0c8FR?84jA$bC{pOK<}2?akHu&5&YU8@R?it!KW=3%MuS<g%6%_b43+~?ljor$?{2(&?Zxng{|Gq<jNdn(`!_oiwc#Bn@npFC4y*>WI}hre51(GU>BJ{zdc`xA=CzwDO#_xdullN{gSNFMJeouqY;;*eK$TQZ0r|uf5kRGIMqI_k5+x<fN-R%Q#E&CpqmDi?!Q=mzj^r@6WgUHGG*z;b+B&5iy41Nz%$YZE_L;Q=^BUS`r7%;%r?cOXWcbU<qRcHeBwT-f1Maa}Cd6b_0cP8I3ofhpN$FnRFxpaDd0W0ATwtv|MA4BL#s%u@w{#Gr3)j*tTg*xH0mXvM)9HoUAlPk<#8$TFa(6T3&sMX@NT*B_&PvmYF=>ojO;d_UrX?@_q?mHVy^IACC1D#YP~+z<r@#K1;WgR}%2bP6q662bAeEe=OG)KNX;(3Vv`no*@N_+k6Uofr0(-U?@J@9~wcn#Goy^B1i8y?8=cHmq8>1R69l(uA#2r%fQVV{FODgF~$W4zMG;@`a-|Y~+J-p5SnZtn&>6ndBwMgj-xlMtO1FjB=`1*EjLfDcj#D%DsqHF+uK|v%!+&X24+V#?~Z|4g@Az67{N4;=V0n`E69-M+=UHUwTW(kgc2YfwK<p^XAx$y3Bh#t5$ZDq$1$QF^#7ZW!7Af@H$m6HbyTu_f~yyzt>%W3-4=FO<iEikI{b1B-R)l)Yj$A{I{PPK*%qRa#*;J5Wi{BkH5yQqIq_Oc1vywOnYl>(9oKUuy+5Hxlv2OT!=Vq+SXLM(lS8Q>}uW>wgR^hV@Esd>y^o1jz*z{5D%8#C#`J{|3t)7giFrtqf3LO?tCt*5nLG9U`s00*oRC~4^xM=TMVhaL;XI4`$0bky)H=sU<nyQ0ZDG1r%Td<mi2u9b#{@x8LcTjj+=n9J0B=SZ)yb#x?)rmV_!rH8mAfx*n)77myuYGKpm5IYu@P&x-iVCrDRJ>+a`l7eUUOZ$kd0#b^e2~^*SO*?!g^h7`oNL3Aw1!+_vM)+$)zZ$hbt~M+OmU?#?mhI(&sx|SIZoAOdi2(2^ERQ-q*VKv&5wBNMxIpQ^!wU9mM1dV=?uHr^wdF$#96-^uK+bJi2c?Qqn}u>Xfzo3+9cY))we=3rxlMk25CA$%qM5iIQ7khkuIXJ%D1_o7h+3=$bpG@eo)fIIL)SYD;$+eap1uMEjhy}kHtzk+{)#Pt-~%`m>RqR($up%kO4tE3E1|jyjZ8^Sxh=`bk>>`g%F}6sv0vdTi=lx;%nIdGe#^IF06$y|5_DYSPJzdH41w1Ws1O7GvB(8!v?vhAn}e>&6XR|Ej3gn%OvKbad&fzRES0v3U}~Am8o#g{Yt}V9^uf05HK37c!%E@GRXY=WM&^u~VZ^c!@bP~p^HR+r2fz}QUhokZ7-c%sU>xEJb{#`p$7zCD!gWzsl3N_R=4~MNFB6!3aRXp$Y(iNwQN~0gyprfj|FTX^m2T$uH3e#Juc9Y|Z_0xOm|Tj2LjRy1`xU5@6x0NH*u{J9x6~I<#m-27V1H=jQE*>_zBU9)&Tfp`RBW+B0m%3hXYvzJ3jDi#Ly%+cmy2~|FqP0D|1s~y2k-q3Q%-?%I%{?$<MbDk=aTfL>$fOs3RFlJ*1c=v2lw8bfedS4<|$q={S0iG`)`rc>Q4&!PH24kj5oOd&p4A0?)@!%P&OpZS%3!OLS+iDCAxYJxCpa~<}CUEwaD)O99-&;7DkKZ5rPFC-1{eT%Ky;}P2Esz&aV2Z|C;{0dvNW+dw)OkjsOuR=!t7~L?{||V+QaOZ@7Rqb|2jPPoN;F7nXy+VWhVo-20!<#J#_=I|&2?RUQz#^bSoPO}hr#aIp2agRRHi&D<Q{|CcGw+HWL}rJ4mfJYg&c58nG3h@{Dbd++i>dIQ9X>(6_85AJ<;Py_1|)OTj3W?*vvtuoazto-<cUq+ytg^M)MclmeTKEVDO_5$BNxcB#f8{pj>crUV-^BfAjlfam{7@0#rl<^(Ic?Cd!@HW-C|Mt9`p^CDNF^rHwCK;kHezs7a)_@+cTEMcv7mRTRlo^1RDRdP<rr-d9NG+o5g-$)Ekbl?v5AiPL;kt_}`PTpxm+h|i618*J`$D-du!xDZ@cXKoxx2W_?t1I%>mL2xUjC0?3aBO*VMM6m23Le~bm6Xda#RxPQ;&a}l8bs*;*xq($_DzCp^#CpN|^`VTrj8%=DNU6>`<8LV~@ccz@$^s#rtn3LR|bWfBDODY8xgxjqWx}-0zTiZdUZR+FaG|IMY3O)ZR6qJiNEy&f5=u`QY9=v%S@ttwg`Nn^oGNx|gpH**9-ozy0UotruQ-`PS`VO6QUeIOjmhnoh5Q#lgUg5i-HHjvTI0@xhOZEd3SlYSE6X@-DY8x7xkV$&pP_oT93=ZBMAWlBiJ=<kI)JbTm%x4t%JTd7x_7t=$M9>uKExK-wb3UsmnWWWBXbx!TOZHdE8m0VdOHSxi?7kq?!)h_R)KU4rPo^9+;bQg#?}?I<tI;Y-QTrY*5>*PqvRSbo!~nNwtO!Ab)iby!(ojPUHx<hyk2$`Gt#bYvis$wm8vDIKxXf+JmD#hbJ&`bil}7+~IPGr_r-?!!pHRV|}LIHiT2a-@M(Ueec_UFtO0n2dOl@uEm30j4;_@8#GKVn-wae72`fF_7FFTt;%z=Gy3elTHrs;<8SOa+aU8ej8doojr#~XA?oG(h>DrFPO1Ta-osS=gXJ?N+wvV)bZ?V+LZE~ZU95Alql7=OFNMkGFpc9E;pjv8&POw1D<ldr95w(IYGoCa|B{EYX^IibhqVSl;y7_O0X?s=#ps~9AvDlg%`JfG9!@r#!SrM3b1gP!(c*lpdEtnfh+sMa&s8^;PpH`P}PUh`eonB%0V?%s~MEGZDeb1Ya2zIm_uD|LvuI}iup|Xi|tmdRr>A}G2|^D50?&j?>Wc<8a5KVv})hsi&Wk4MA{i*Z33)_nq>;m?bWZ_`ouzJ>e)A(;WN#}+@UnCq#jFRTpUK)fAh&vnudA)WGRl(KT7hHvpVETrqXb6+LB<SW=uFU@TlY2UFnrlaXli+KJlp6bn2}+t^uukH;_9TAHH^o&oVbW<$;Gh9*Rinfa7p+y6QvfUCs80zkP+iv@lCCp1;8F^+GbhhRW))i>5m&6+}abXj<&@uN*5uT0cz57tw|GDoWU(HghsB?NUvy29fbFhA2}Jv2*$sNq1WQ`Z|+GUkAeUFza2kL01Wc!h{0=h*T&VVPKGpyF>NnASD%s^<vu91jA8Pvl#EySShBPwd$3Wwz6I&!5cb2`mw5w<H@||U&kzlIYvQ2s{gzw*=K(#uypQ#1>tZ4OGEMKP6fVUsu5xqgB5Wd7&u=+x`l5fP!R8^Sx~6>XfdH<mt(Hig$#NC8hzR10vwbCbxV+eCzR>Q!Ebvu=m`ciqX?<*K8eK@?{)Y3HZB!M4oMwwK#R;;K_gP=SRz+zml`LY*vaLUSux7X@;tvo*IsZRgv<1dhR<k2V#L-}7+AvyuQU%)03o|{1{91BI}-|C5-8DTZYWQm4Mi*)>ll`H{;oj0!#c=_#;iA^t?6gjjJr&s9|d{qEi$a7ZSfIv1~HX=z0#IBZwI*pA`NAfj7*LT`Uhd>EiH^yB2LTwxk+g~kO!(1QJawpbR7kZox?y3$3gB$%>1`BwxRE)Rbi+q2=y7cEqR9fWG}*3emSrJN36X?VJs>}m{^)Hq{J+tur%m9n-@cr*YWtqI5j}hx-hq<ay_=%YyTHKS1d9',
 'diagnostics_snapshot.py': 'c-qZe-ESnvaewDu(WVg0&S|+kUMweAegNmY6MV5gPA4ix%n+m5+1}j|XJ^*)L2|_`kU$Sc9s=Z$7=d93(OD1_*)d{834$CS@<7@D2lJQYSJmA!Ju|yYN*06=(CtirR99D3S5;TlXmfM(K`Q-R3SVSXKZ?2{8I7VamSPzC<2cFkFc3d}{6vh>WQsvS>Yqg-E#j^S;~*-AVLTQyKg~m*o3q~L=H^B?oh4~5{B(@2Om6VxAc-Ow<Y5wLLJxS56mc%ot{BRZUqtyZ401C$$WGO~A<)aisWg+!dsooE?<cV|QzrQ|GS6XBKJiZ!4ti6W`<U&Up7*jOR%^`se1Z{Xi6?k9gJzMRkCJq1-m`gT9^VgVqcD;i8ymo1fR00tKz0aPSDg824C-WVe?#CePv`1^{!*D2Y0L=pqQoC&Jg3JXa(SM2WE><!+kL0VM_YgDxGv9?=YgE%qVwk4kJ2;&;r`r@3dzr|@H6VK7BL9){Mk&#!_LUL;8kLODi?imfx!#MZQ^vXXznLkzM*g=;v{F%_2hY&<ypsV?eOTBThin#6Z_`vEX*frUIN70d!?OZIo6+6@W28Q7t;Yr3(IF13yB7qLL543bXV+H7-$@6a1Z_AtkdXhamD^4j1A!|HW4(}ZOzbpskqT%hW)pmF&Ao^4Gw8~5?Y!+I3V6&p3AI5ja@McWi&L*Q3L_W!=(2%HXn~4zuAf98R5V0Xqabdh|`Bb0M&blv-cxzD5=R138`#f0q<f@dFp3AALD%|goa3E4O)+XCXgKfQXRxPphz4eOf3PmW}v5YaU$n^bts0#be45aiJn+A&Wcoeeinq`{;PhJNtZYqW?`J=ejG?M5bVivXcKJ3b%n%0!97d4P>K~+?W%Da#UAjPLYiq2bW*F@|DwhVMW)_Z<~+IM_E?f-N6GP47LGw0#Vn{Q@W9K8>C{i>oumMoadOr=RPt2DIUN=2^kJ~r7EY9mGlw3+_%sQ8R-`>1;aFfa_o0#bz!BXo`fKpm7i|-mcN}iQncLqN4$pIh9}la39MO~)(_`2IILEdMP|LN>{)IE}voLe|!YN|^)DI(n5J@2GV3MTfA5O0R1NI}h`a)oLBA;Aeei({z=*J?szWmfD=yH%GCt`Yi`HOH@oTv#|F&C*X^6SfIf%x*-_2tj$`PU*tgE&jl6G)Y_s7L`yC>)Qrb~NTC4Nc#XMOQBz>K)F~WSq)u87<dv$jn;7o`!}hNnJ;P$z3%me4sAA2469&!PQ$V1z@yV#nwy)iCUi8iIhi<jQm+9ho0`!RHKu(Y+eT<FSbex-Xq%^W(k>BKksNZ__=fJy1hKf{m8)8ngpg;X2eZHU_ycrE!o6p{z9=nNT#z$=5mM$YNpzI+3w|&F!QQ?gHd<&0SYE^SVXO<WxLVJ<T~wJD{Ys%1y)rv(cW|%h`S2SH^_Cd&r3<M>forwJw9|6rrsHRHIHaW$EnWDzQxII1(K<YJ%hY`VUZg1M`0Xh*cdu+*rE=x*F}4ixqZk_0ehw2yW)>sYk`Wae(ps+ur^%5#Ih4$D<bwQLR)}}G|D_^9S<tAU`3&pWn$kE-@8i-v#e7?NlJUK3PA<}XeERl@P{Vd2mp>^Yu8i%yo04H(b%eX<(7bVD!|P2a&(xnuB3&QE@^b)p{>*=GO1}rdj34n$}#%=Z?7*u&b3cIzJBp(EaJ)ai%;@hr3oj=)kl~^En<j?(3t7fuVSTZ!|NCS6RUN4;?d!K5dikzis<SiL2!Q&X!!BC=O{FdV>l?n2ux0cNq8!iDbp|7h}lgz9pp{ZFo+aCn!!AlG{SzR5%AhEZhf!F1MOO9^;x-)Q$G$zN*7dqw^qZZkF{^xHu}V~r509otx($Xe1_W#YlI7^QJ4$Zh`wkjVg+KMk9ClwbWBqDlLGiC^t)yYG>hCmo|e$r>=;D!Ovmkwqh!!=?x?M}HUFB1)VAO7(l;y<Lz1Uv)ICy2c#cCR?XhZ{HWOqUXfoLtLjc;XHHHqDE55T|4meW1ibAjvG~l?t?hRLy#}BvPdHR}&#}MEUw*SSii_-SzSHFoT;`HjHq%OmbyMAYMR^hi2``FF2wt#onc!IRkP7M-mqsv-}eul}^4@s&>Yr;V3<~+=mYXi-i7a2WG@_go1uRkgLD9q<&OMMN~JBs`<4-Y1O>IHN^rZ;xGgFH$3g)SJ*68+q*Y5B$qM#*y*M9e?)h~(xuoA~!$`5tcti<Pf+7&0E?{y>BLAb?`Gwr6B~SyPP?b%^B{-jsI|#>3jwa<2orIzXLQ;7v<|&Zb$<ESGZ^O;I>9%sl2}R7tdMc1hBg3w3>58?eS-Gmzj%xaitzm>oK#58S07h9M@bhoOe70RqmdyEy~U`o(Oel!0Tn2S<ov8S2tgTf_`<8(xd0;><DD>$<gDJ3FGvVP3WUx^ggm2I_^wUX9IwuDEkY9b<;R{1lrbs|4Q9ajBc-OHQVe6n9bH?$n-0Clk>9$e#{|K52MX_X-2I?qb66%4(LOep6tGN&sm~6f|o)@!B#eCtgF*y0^BDHXqOwV>U<aiwQ4RyA9~wvYs~D1P+)`v(%V|hEikcpwDP%^X9T_rqW(+t<Mrcmz*HJzpE__=AJr2<N;I>z>W?JwxS1$K$WnR^N-5aD)sERFg=-!`1FF3FYk+;RyaaYs&5^fLS4Ua0^OK#RC&MlQq}+j3p%SQ4U}+4K7scGbz%X~@dA>p%2OQ6n4@0Wmyao6QC_0RZ$})~BTI@-9C$XWmZKRCJ+=}a2XHfZVi;HYOxo<XhPX`#$qZ~=3!pZ_t-0r#D}7dJimilrMZ~_P?D#ICkB#ooZ_BP9<`Sc!vKP*AN;Rc^BLel6ekC{Gmc34mU(5NlM3P(^qdJ#wv(9?CxM@90t=(oZTMMd{VyO1pZ%ZBa#tPK9OczlT_mN^=r#Hxs*{f+ahtn7!*sJzL5}@1dp$H{p6l5cfL$-{D;sAA6o7OC$QA5kNY5^>j003R6C%s)oUPUU^(KWiG^5)CwCZ6anEZ{042*@{w2H7Ekmv0&{$voQP93WIjH^{eRxP=plJZ^L^J|q7)xq1!+wF5mv>&Kz+k-I_4#Lp(Ut)6X@*N3;cIKO%x@HO%`F$2Ke(Od*Y3`aW@!@|B3vP$LXC^;D4hO_*2-Q~Jd#tk(WtF_7v{`FFyTFcn2J#5z|;LAk8Z`{fLHvDW?M@wa%leMy9_*$-)Me!+zIK-LWzo(|FlRyDkHRs425@l|aQ<<KIlHvtNmDuln2dW{T&m^b+4)*9@#O#nx+B^+65B4bOCrCUpneT6MWFD<7f(H-m1ES(6d)unZqM@^w<#RL*lHvSfgnMOw=atzxB?57*w~BBVxPoQ<d*7X%?^1$gYa+w(1jBl-EOLr@UkspHWV%HuP!g%WdDuM*hxw$xbNB8aEpU0yhk9gxhZabpa423H4u`ule@Mow|0h_1q-P7_KaTs69Ob*93X-&bq}yP27l1>=8b!(3)>-P$`hGk=o5)lyz@dw?3G~rc22Wk~@y;U_n&cNjkwSg-XGuu<e6hE!_HL15<m^p$mXS{U?_d9;xFGuXWES`{jt#N*;Q|ik)Pc+njH2q-XEhuechM6+up(I(wr8-|zOY<~#V&FQ|IT{r@<TPycnXW&-uA4{)!D%#;|O!u>&s_XAG6APJ{P41*%ihE<}pw_;v~Gj{9uY$8fPFrSMdj^M9z~_E(6JmpXVwB$FU1-Ted+Y9{d5k-9a%fH8P?58G!L72Nd|j@L7>9L;1_Uzxw5zQOS&&2U(tnCQSI{GmRY@6T)bhLVy2;H0br^PYXtmMt-1zCo|f?CjpwE`3be3g+MA){6uQ-%V)mA4>W0GGr4zQM=MvyBf<T92YZmU14q0pY+hT6LY?&+^n{D~4iHIY8<muC<z>(1R76acZTw#T6aCWG5jg#jk*Q7QGYXxM;=jqk<A=}z6Mj5Q0fS;rldI=p+4|Ol^6>%5nO>r7MCeLAZv&!U=(u_H(Edwx_23+$$?ce1Zp#DhuD-5ymt;8|@mIp>y@$hmlhXydcAQ;7{Lx<Gk+z)2ZauH0^t}oOI2t1z<g2blMpv6VyhGF_;EqNI3LqaI)XqJvtJ1JQX>-#ht${HPlda6pDmrO7AfRr|IG_|Erv;CwSMhZS3lK)x!JgJ}2ZuNS&%oA`t6zyfd;85dNPb^@st&-X(0F>764-7{UX_I-k4y>S-gXIno9&N@43NV%r7-izah#XL5_Ji!XY?Z?>#tv5{<C<u+y%jU7Z=`wkK;IGmo2{fEY=Ey1RO{AXV?Y!8TU4}#Jd`AjupS#uSG{uD`SgtSG@J0%PCW|82{DErfzdS;D#gvuW5oMs35?Srocweg&zF75W!rnS?T;uWN^Z^rKSA9H-@MWhK9%|iz4VmmF-l%#}`L9+vKGVK$TdGNnh!+jg-gri~k~hMw>6lvt^*#3TU7@yc|q^|69L-mLN8)Ql)ywIy$9bHOA^*!Z;_V=Ld&}PlP(56tmXKi1Rv+A8L3yh^GMtlY+<KcMg!Tm~c9VSy#=H@oAVQG2Ms^zn{$W2{L{H2>n#y^EZW<z@%`r9BzL|xgSNd96-e>nFJ*|RJO1U7^O|o2B5312~_vN6;A}HDh$iCeHdQ-H)zWu$aXj*P$W)77hk-xgi+gy5CqTehD<yUAR1TyG9}*O!a-cbq+OPH1>UPAcoY$nL6SF?-MLGEKQ|(VUp1<ipJPK)UZ1Xs_7w4Y_<aUwB$+^(<+@~CJ|UX^tETDT3W5=+f4+dhfAOF3Zlh5d+u(PffsZ)(C4uQ!BTiGdu(UE25zwlXqzfqhF_a{|C@l#ka!C>KY+~FLLc%`hqSM%n2nudc+XYOwx3R%~3TOND;rqrM7V#7_6bRnD(opgE>X+C$SUVxE0Il5(g%E3Boyt<vn#_Qx*Kc9(LX{=N%u4fY4EzMD>GB_@#G#M1`v+XV@V7Y=8^N`JOp};ubyud7SazXVzfe9<t)zsvS5tsJrd>q^;(_ur$^)Z2Z6~oJQ~kP4+%s7%TD($2y$#Fb?cdtv?Kgei4HzFbM>^-)Lp>&`jOhuKPDj0D`-dr4S<%haOHTMzU&tgCI;sGqfk4H399=YdT()3jj(hoZrmeB1o+x(d%vd*3JZ7HtVB<2XO+Zjj*H)s#Jod~3M>26oIq}CF$+=i8th6aNt0@0$?!zTHv#+y!$xeNoZYbW=r`OqhTFizc?un4qtQ@aa3801iQH8<rk$$8^O1q}i?5d+jRXGq{8@Oc36E*aWB!87E47hOcE2C28V`}^05=&j5+WT=hdzErKB|AX`q8EissXl{h`O2}6pLh=+z53ey!$%Lf5OV;`?4Uab-nmT`?vdB#JbiTk;p>ljQwl05&!=7s=QL_2TL5Z~4>9R4W9=xK9``Fkw7nq!RF2zMgkk9zB3D&#X)^n|=JEzGmP=CuquXx_#&Tk6U`%9tMfA<Rd6$(()7Yl!XhFVZMy;`M>FHEu9A%PZl4O$7K$;{+{a2O^Ses>2Ll(>6zI;=XtdVUi7d!r@5K-EC*xors1_#9uS1a_vINrfO_r9xt>b1;VZx2IARv9reT*m73U6<HeQ#R75Il=R2v)b5#576ei62{7d)pswGu&wT~ckXbWcCjuYydsxfFO*!xsO5#$ts;;*88|!(THJ~PEf|5HNj7fH&6>-)Zq#qb?Xb2U&}cpGt}Iv9D0)T|06_;xIKtd)UMwYIwe=pYh_Ypv{VaT6ioJbX_?DB@)*c-T7L?eg@%}X-w_7xHmvae(JZf!?Y#VtZ?OOX<ap_7x*-1F_Amya~36^Omb1YFCxZ}aSbk^46R;3Rsz7gPqXo@OKmdR99z_KP03gnh=nu`P}&IughrNEE!@}w#N^UoX$g34m*@v2OtzfwpHJUZl<VS}f5Lj|<wqe3#t!t)g<mI9;U;lSBCXf*~FFKnq?$<d0lnksF%tkmTCW$WVUpLI=DU$#xrC$krKEf7m5Z|yHZXzgU1L_LJ<JGWc&>(Oh~NoYfDABS?2`baFP@wmk*YgRD%Smc`^cv(1<oSId7K1t{8<+PP7%ZUK{M&HGXo-~{n-BWolRBp}}fW%5Z-sm1J3p!xFo#Ikx-w?X~ioRzv!c$+;>ivV3<<F|<{>E1+eTnr<l%hAAv!h2bpk||r2sK-+;84A5wV2{-i7-jY0HR)eAe#_XWWhtMByM8{^_63mN|Rb$OVne{*3*`kseZPKH|W@!iczM|;rJOT7|wRJD7>o7m*Y|GD&Hd{1dsbG@%uxqw#He<%n0kU_I90cG!KB17Ua(3Q8+Fev!qA^Nv!1Sq-i!jPAW-ibn_aSc<Y_}uRT8e3-9TpCvQGIR2NGlhxLn{bN`)(j}Pku&1XZ+L%U|datd8~pg6eJghGXuthXpo6F*ibnuXu8_?6Mh)govea&@2@eABJCRpeZlU2d^{IBT=kkdvG(T-@hvnw50}ZT(giAK-(SnW`+fc_}jq8iGF|Xad8{A73;hAAd6_Q*hCcu%f`HHn?Yu!gEv6W4eCy==<KIr%&H}`ZkN+T8ijWi&LU9!gOIV)ke!XL2zvBo%)G4ERuj*|DTD_OX7!8&L{<e<XkwWY965qRca+()Sr>04Je=`>7%C{UzDrxu{Q5nZ`G|;UFX_3bMaUbp)Nv6a7jvYW3#wzHI*`0eonER95uS?b#8YBTuO8H)xWNyUcsK~Zgrs$L=d?8V4`uaLX@8`Arzr#SRDRBv9vc$XP}x$0h3D3nhInBKS~Z$__QfLiyo``D{AxXTm^Tz$19Ctxmy40%5md6awi=u|8~6PW~vW&GbvBZS6$2awPmx*b#KKlcHKCIQ?sjGvnGXIIu@)lB#RzIz5`kRI*QBYA}g+2jW1%98UcUfWZQ9j^MSI`WBRi#(p!d-{J&*LIjXPz7Y)7p>6q&)pKwoSh<rCbndi^EJu1HRJlC2@7aM#5rlxe9tt|+Pqds(k?HpRDtQfC%xqJ==lI-fK{6mw?SD;l<bp1;-8q(9>p3$d8HX&igu(0JoUpNMZer<T83c0LAv+BER)yU-l({#LW>7#Fmu*Yc<&*M*UJ&)+`c}`z{zedd1_&?lHDA5',
 'export_snapshot.py': 'c-oCtOK;;g5WXAme_-7MDa2NSpt}fM_)wrJ(4yD|Xto#EAQ*8Jb62mB)Fy6_|K1srlI?gmtqwNDdGec?kDAS9*H2w*JyWb`>rNGZ&VK!Q(OGa{znJD8w{6$EAK7Psj#+r<!FlGbD!?(*;0`Kicg`Gi$|?`lE}zY2(}}60Uwe=H4#thGa@)$>jqjga+e{~=ZEMykPa>8iH~2c8jEdb(S=&A`<xuhUZjM1PH6Nr8WHp^=C^0>yC55O2dC!*93B%`m=Df{aR~cWK2As&4s$9#$j$<FsE-x;x&Oe@ATwJ}wQ#zx8eNmOs?6*Jv_&~-44elTsd35<`{5G5*Z#!rNG)1dXX~lbAF0Ob+nV}u1I^{r*1>f6-ttrF2YL#|^^2u1)T1G2^I9f4NhMk;(cOu*D4_PbIh~a1fZOc|8^7&8pzOSM2H$fsaxB@+R(>5z9bz4Z89h&A!YpIfr;EP4udp-}PHx~5j*7lGLP|IPYWEcL(65)(;v_vu0QP6gd?@8j?-WGtug94HEIpeY6a#a-2dC++f@?-~39Td2Wmd|$0jpmyrvjHv}unK-(qB0`Zw0?+1h*Dvahm3u<3L8<J9acpd2kbAZ?lm@-75uvH{4R|GInq5gN`c*u!{UbAN&}T9JOt%6XS}x+{gtLsl}s-lI}XAEo(NX1TPh!|a0AE6x0YVI{;o1^3s?$N+vgS^Zq=itW+PQvyREu>_kQqu@H%E1DsA+bEe;Lkx_<r)JA^R52QP3LTMrk6W<1f$A-oo$OOVY;ZB?oIPAg{7N2~{RW`N_kCI|RpT~PO-$AR)Y>Adp2;~T06AF9p?Whx#j2BbDa-!G4`IIY!(wpk2r$P0wpMV=ZCpR!0?lt;M4K314s5LuRQ;Ypi&<UA1?CaDYnZ~ilzThtD5SZ!Rbe6byJaK#pC6~-rzYq8#Yqgu>KWE%he2MmtsQdL$65~9qY!ayj-Ch=12^^O!zi$B^$3Pb&U$dkfH(I%;Qbyq=5_m-A9aklJsl!2gbnx`YJ;@?0j5BZ}tm_o1@;djoCXFZWO_A6aNc=tFWC;pG~W5r&vLn4L2HG(WS&p)+Dgn>(SUw2N}IfKUa7Fo#^#;krpl1FkqWBl)i?;Dho&q(zZWv4*_;wU=!Lmt>F9^~sM$a3Efm}DahB^T5`80t-?xk>k!ZTm(qsO3)@$CTe@&4kPU3Po4h?{5XdLi80e_%pIp5$*%Wmw_}BWY~(*?>fL;9vYA+XBa*i2cP;*rVk1n>?NZM<do48f->YIDIfU-tietkZ5R*7dt0S66;PJXFE4u@YE&JrQgMva{Iq@th-u|%JQKp?B8v@hxAQRGaeJWFDfVVfx|@JoP)(STKf)|=$MQA(1o<K1yd1YagfBH&tHzWl4`(<Yfy34hDb4ZF8q2o2{C0nLNCaOoC1wa;UnP^04kTsZ6e;Pxl9DH+;>X8y@-L$=LTC',
 'postprocess_snapshot.py': 'c-p;MOOND85x)1YNScG~!Q<{7Ew36C(kwfxMJy~DX1UCctSmdLE4N&>Co|pMwXK%8aYP7l;<BUB3ULDo390A6VG#eL`6Wc;!%jb{x>^hLrTxf^j913@Ww?%yk3U^53tlopo1Ci?c3D?K5nAR%Fqt)s)HGYuGhUv>gl!fq&sk1*Nz{swk3S+!T~stDnV=U1DJ#VmRkcozkB^V|MoU^zxhnX=-F_jf@@Ofl4aimag4h?ofLk|6t{TOQqobpoEkO;Mr#LOd8PV*Bz@K2M5v7)vpaL?)`DR3-XDD{5LRMx~4yw<CMlB!yAc!I>^g?jOI;k-^r$xi8aC}1Mq?4$hDu&8Yv4vOoG`ZkvmC9zh<d=F_Vy?+|NRX&a9So)=W=*Y^);n)DwG8#b;M!6)f~8buoX<a^g@k9k%vq`Ck0KQLc^TlSfjxsI1gi_0F{`yPb;%2sVibJ;<@}6E1(w+GByvR`efN8z5r~#~<|kFn$}kX%AR<%}sldS-BM1MMl^~0v%GQ_`V4?^Mx>@9OW=HCHgzud`dY?QX_%DjdA_$^RLZ?yE)Hziw)bb)f)b{;~U7D*fYV#wjL@teqRih5F5OvmJ29+hG9LqzKK`4Nnkibd}qU5Zo7Gdxp!I(h;bz1i|))`RJ4NG*8Sm;6o!CM5MpW0YWn7UO=Cb%WQwr~)w=rA9gGr^bJRBvH4AtqVXluBA{^>c7Vx0QOf>erCTn+@iWu~X0EOA_jr$r)2&Age}XEZxw$h8#@wqMYSmBA2-L$#c=LUQofy-h8MLDHImH1+<M!-d~7h1<RD4@U6Ku^F{ac<({cowW$lXXRa4qLAhk)8I0f?_Ed;UguxT-#%%Skzff}d?sY~syj+og{_ghepU6)>Jo)U|C*=IyS32Qf{>2H}nFNCvnF2czByhIVh-Mk9wMJw<C#RlPIuG<M2yitD<EWn**-)STP~$)dvUvx=EBO+WXb@&M)p#nevStIdNA^9{={53lx$TQsvu!V;ddI;-+;CxUHvYAeQ=4NPHgpXD#Kbu)7n7i-;#I>`DnJ#;Ai?V{b>KBxQ%t^$qC2Ac)F9q;SQoc9Z%T5u;>2mFL#!8ihzlh_S5R>GYx^M6MBT{5Dvb)-Ct6Rx^wpxOmJqML|FH^4n09p?=?5rT-?j4;74QNkn(*R)_8rC({cgr2T&5^5j4)94nN$?22mDJV|B_*hwIXJ9U6H(c<;A%xPexEQm}-h_uT1Xn`!e_##GL5mjdRF)_3m%w8T`9_`^VaPWJa#gw(FoP_pcfbNQnu)HQUJL^!(>l=`b7^$TRi@>}yi6Gn#E<vZSDUjz)m-1um+q0k29BAl6JsvS1KKMyiFPyo3@5w1y!BbHHQGX?ATqCAOWx=vy|m1?x&G@Kwg7OmUFS2hY<GISd?^yo3QIAh{M6L*>uZ-98OIxV`x+stq3eGyLySI_q6{^`<7p?c3ix#&I{Vw{Uawnv>1#+pjb7{eS=Z_diVfDj6kgfVJMHEVKnRuwHe844~9h%Dsi%=9D+l7OAP32U!2aH`navRrT&G;HtMbU#r8+HKh)sr?q+!n0v3_p+8(b1X_e_49YZk<8;b4H?o7Cmv=(X|I$3r-t89WT--GM#l1m&;Os}l)%}s(_}3Gl!Jso6ZeRBBpz}jaJXc}aw;ql=h!_=@UH<x+Gt~IXmX-DG%|8@*)j(nI?T^F!=gNHC>K+DtNCiz>o@Ab<t|-Q6DMjln4&YdQ-wl~`ruhq}h*~_LZ|RN-%>9EvhZ#C7-3G)!9pv2QWT^qreD?-UhIFgp^AFrr?}L+bMCSYYIa=OYdH1gQ@rq^Zy5i+nJ0fXg6hBPAoh;k8V1mQ_h@9#J^)LWCIN8AD!JvDXJ_wPm978YM%k~NOIv}S;bxTLl1tk3Rms+8i{0h&y_WGx6Yp<SfYxnSp0#3DPl)2ZZO%Sr5zdc`Z*wPKu!rPm_XNYgV1!EVxk#MiUME53OQA;;?2?unp4{F-z2Qe`aIQMFkp%#GYtkxU^4(h0z&qo|7xN-<OvvS%EY(Z~E?CF01hW7cjW9;6V1M1-pmcCZrwNnlb=b^r`XH2gp(mNgQVlBoItam^)vA0C6DN}E*ai1xTpu#1~Aj}<E`wZfWEhR~T@SAoL6S~|&PXpW#<6F~p5^M=4gB1+Zj;n-?D-(}N$ske9^u2`5g<}|A7h6ys31Hwxvl<2mMwMVp`K&o^Ut9H4S<+grDwWh*EYx;7Ts?dUBjKXer$;?T#PL-FN~&#Y1Pn>zfT>`TK^>p{x6t-_s=Bqzr(cDvDmKMYm4KHmd*~M|lQBl1XN&Q)j%={YzZUWO-t~N{?@X!%F6Zu3#;qB>eTl~O7_3v(Bwuhh2J~GuFv)KA-8S8;6!+?Y%jq@_G3k|-IhwVDcuZ57`3T7d5VlL2Pj<#adfW)uQXlN+BW|!`l_@?nUTmuA72P+8Hc@s!G^Wgfn18f;TUZqRfJ*jMU*d^%2#k{fH${JdPd)@JgDVB#3!R%RO41amTAE%1X@dvXzETLgrp>`iS`=R8^HAfCQ*MhF1|H031vtkyrODxoO007(G{o{eMI6=_s)FM$RbW1Jf7m&kjjPgaYU}CPNt5H0uxOFyoSv1S8qefGfDPyYYy>*%iYzK<6jPC{_&Ec=p92ojCBO}##2Sjd0xjV&aa}9j0nA$R+3O#)#`?Gvn@js9+Kcl&@G(2<e7fy-ch6xDON*u#xw_uhu)smKVWCg=ct6oQu*o?XqMhg{u;V9#lgtjtp3L?M-eY9$YugMOOg=)J3hG0Fb)U~k2Qp)S-qm$%h}_Y*+u#n{8ppeKpY^eM-pT$pd1gM)Z5lY^g2+l|6}dp3WS9cEfI49)5$;n3FYwGxy0paeJXBTD)GFDNU6wFRCD@>l4Vv^ZIXx)TM$~_&0$rO$R2S&F7cYCKbN5xp>9^}cPJe-s$)<GR)Sl7Z=fJ_?+S>tU+GNkpS3QBxD8QoZ@Mu5|4&9UHd!>|P7J1c0Vb(gZ$M|N@8!bcBCGyDM40<=0;c47C{b4t55_3C<goT<-C`=MK*4l3S`P>_7xIIQn3pORC?M;FD?LvV?gA}M_(MTx5uuh8*frg0TI{=Mt{t!K2$QV$99-^QnJ%nkcJ$PU|?0GVnwL5%#n)BWa4|MITPF06v@OBJof%{01n(qTM^5U*94KEFovE>Jw5Ig!43`!DgC9~N6LuR|<zwtTKZc+EMfppkObN7+`qU#Vm)$$-o5U-C+G5oN@cVa(-OZXZs9dHw>stwPOy*$=-!RL1K*;=|EF`RR%516eQW006&6F|2F>~zhdBOelWv$Y9Ju;VavaMw20RI-DOZ{$Yjk9kW%P6!Y|5TZX{UngJkIvD-p!5^D+gVPR5CXU66h8OMG+x(~_+0Zro%)#h@t_#~$un6JWK>Vvpfs4f)*MZcAPcZP40obtZL*2WK`U|zcez4M`o%DJ3e{%I5+}XIL=K}rX5p+V7G$s5jXB}?1rk828_A(5%<|v0I+xezpHXQ4>Y^<}6{sZF4s&o',
 'priority_probe_snapshot.py': 'c-qZ9O_STka(Dg;M6Xf+Ws>_L>qVwSr}S2Gta^HqN?O}5wcbL3B)Ei^1ZV)<4{z-`=NxlORVud}a>yy~ki$#mfB2uIyQh0F0}QZSNiP+Z%S#MqdS-fhdirY`4<0=Du#Q$MZ?m1QlG$(*oyA@>o5e-p&9e0-%9A2X$KEv0E{iz#mT@$X^TJC~?|uC9HZICy;ANXK$<k=$eTY|anrzpj2M-?f7J0S?80l=A=W$w&7Ta>0$Ix`W$@0=WUgiMnU6!q0Ud6L*ndJj7DnjgZ6_@c`w=d(Xa+OS_UAkSZxU|Pbv+{Zq7xHxWOR|yQKf&PT_c|(<@;fUO_zjd9u-_R~@>qTq%Wavg^wV~_$umTdF1gk}%Ve#al{CR5JyTt6^A+?o0*n^1^ao`>Wl1`~AM<#TM`tqY)heDLQZ>4cvMtORa0K<D?W&w7v$EIw{^k1*uin1#CZ4~^lPpil>miJ88V@fX`@L^pK7Rof;O$M%gMa>WSlCVJk3ApJx3?_Ut3kBcz`{q^+1?eFd~sDTU9X=0JlP+8HF&u9kbXV!2PVMb>nJ_j0_7m&UzWq8H-lf6+$2uz7T;~oav-7MtF(Ys&*Pzx5w<;mTDQGk59a8lae0~L=V6@ZSstDz={%U_(d7zp5MRyuV}du2%P3jF42v=kuz0k}F5^7t)7xa>B}I}JWt7fhd_C|O#wh1*M^G9Q{T8DSA6~!2-WWFi%VQwg)bopCrAFd^4B(?IO<_HcfjY8n=>R`d6~a>0_^3|a1GTmHfV9{EAEnm;?PW&eBFjB`4*WD)$FN(o)fS<vsSiB=3@Xm&d6F(NZvjo>IhLwlUYU8+<RbR_jv>s`qJ~2l)jET{o)nA#0A~ybM&r*`SplH=Jq*RWOv)up0=wG4yknz9D`KHts328+8z2ExT<!`-RsNV2YF`~|VH|xj+SJaNErDUj=~)~iw+@&k54>qMzh>CNreWuw%Z?a(7qr0V1NZ?`<vOFJjMqibzomv$7n&iglkvg+;g{Y+5C5E68*7$j=Sd8xSY^>1s(%x$idaw*z~BNJjQ7lYa`4q5(AnIZ`txkIUE@ihS`)qrs743=!@9xLUuBB>j8q?0Wuyx0^=1{#3}7bZ5i?x@{}L{W{7t_t^XswIb(zO8Oa?2Buqp5qVBcygR>ccg_2~-h`kp^R79L>R(G{L2fxowhP4@PX$p-zCG%EunkOI!Q@wk}f$)@bnJZo>t>s8!8g}pPvo;QKrX&vW?WgZA3`rvS1S64LW1>Z-jZA>g}pk@`*FyG-Bl-F6EtB>P+8=EvK#5Mvx5cspjaGjLNSyU=E;A`&dO#n5TPJOEnTEAyyQm*2!f7%wM7kTRp>7s=osQxC-fp*uh93Gt;aL{=dah@#rkm_p7eMMynov7{p^ES!j8c|tu@AF>E;Z%-`Wmv#@4^4wbT+Wtw1Z@F>A#A@?wlj!4FMy9lIq<<{%*@f`TN@B{yvrpNdmH2-`327A+vDSR9z%yi3{XkJ6GDVq7fH6oUCiB%WEkcygq>e)cWJ|W4F7JfgFc82ATI>^EjLhLfSr{v6vZvZ9<ulnPE)Bo@Xq^+aj27l(0+~UJ&W@a|3=$#iGRx+mNSgg3!k|Oh$2@sK#+Q(H4YR;h5gv0;cAUhs#eXxBaps@q7gB7M$0^2X@nu4SP9yAwhGs6NaZI%(P+Dw!=eN=j5+JDyfyG1KD_b2gB2b=M*;!KrlOWx8-gqoL*Pfd4x>lv2)67xgeqM{A<iBvVolGcLLseed<};JG7DT{p$9n`Di%>NX)w%z(NH$bWt5a5z=VAS{IwF<^u~!V01GI_M)OsY#vt^=zFco)8x_zgtc@I_K#bvHH4muOi4StJe>$Me6&A3Q(sFe@K_=Cw9n6iESy5Uo)J)Ut5+LJIJ6dPJdb2c{@uK6X;oUmA3JziQWK7-{)p-v5I$MvnX@Yzx*dKVK$9+AmQ3|X+j7s8*IE8_?-}i>trItjgk1xE@f%o-9PhZukFwCPS<b{AwIQW52o#IJz1pH&3U8Z}}Xm*}07R2S_c>z3X9VIC;6BatW>2`i5GK#^yt5e$QhL#MtYL?Cm>M93aV;*4fz<Z>|PgpBfalE1OzIiCnAtCIN1^|mc%ksHn^@|AAJ5U+7rO87F9!o$29y{!zfyK%7Oo|N(%An(9Wig={4LsFdI8XA4uYMPP#g;!TBS4}@sx1$Ms2<icLXj_t&QhS5WLD6jjNBVQY=H3sk;GHjDw-uv<Fh1fBO<+~6R!6KPll0&;}YH{cmR%{!|4g6GZ9MLmNR77B0x?~PVoWe2J@yT2VUTmg^qFZtW4?l;X0e+3&I!hW|;VlPQFDAmtG901R<UJ36(j31k8{WO5fEjufmOwouC4@iBr^g+rb0ILRfZCMY0g(hEUlWxDRHIY%xHsj?l!r_*2;TJy^3P79yR5$P{7!gRK3eCNI(4V4tp{YZ%cRP?W6K4J?csNNfn94$wm2C%lZbNoD$~-X??P0(Pq~*Ka^%iBM?l2>+4**wT5kKM@P8lNAnq)ju4=^B5#qAiSH~z8hwg6gGMgxx#5RiVrHL%I^x;NC*AqKmoh;eQ2`Qar-D7cnEyaCqn^%BF%e?qXHV25y&{eO{Ou7KK6)%`VM?wC<4AA(j5y)qp+kV1!eq;%CL&L|8IM(OCnqRbOc5k*l+5jYYu@~OB=Vc76Q&wu*JAYxRXXvUfovmr&2*p+GHM-p?a)QaInuhCI4|U`uqH&O)@bz%%?)+(`cekC83Fl{Ak@`LBj&TH1tvo0R!*t2c>+0!4sRgky|YaRB_gzg%=J-CW48W!A{1H_xDc+p0+-K`6DWd30g{(oh%H|#1n=);HomgP42SFIcO^Z%kZj&fLIQJHQV5!&=f)%8+<5qtwoa>OAxdrbbJ#*xA8cj>qx2bSktiNFpM3og<Uh#ov1m<ld}n(7BVK3M1(g!YjQ+<{i&k18!30|yRD6R&fHcR%Blep2DCxJ^$*SZN!n6D@jr{&6-f#W=@Op=CPw5$|LV6PU2P+>Xw>BHPDBYiZb<4aGW<P{BWtJm12S+kUGnbkEBh7~Ho#X#fd&-=tnG}vF{7iXh|w0Gg4U8;#4k`dwRGK9hO;coO{h+Qi5vv_3}p}#G$?eM9lN;?td6!{D(eHee$sM?j0_KGF$Cgow5h~orY77EqeZD6>xL?L&LcRs9g+&5Dnu(DpzR$Q^Ppf)pSgW+&s#jW5wSFZWxWk=1i$m-3~DKR;qB<`mj?~#QZp*EHHg>K>oP6^>Idz@09?8skQ-}+-Y~#vK`;m28rk7wzkkQd2#|^V7=dGLlNHGoDHc#zEThB6zn!=S-U7SM%G8R}ntZwcWZ->y_!a5kK$zY${rZ{r+x`7s+n%#wc5}RO+wT2KaKRlI(I@TvDR^Vfd+prKwY8Y=LnQo(WrjWCEpg0S!biIVp1L8(o)ZzfIyTxT(Mu=bIy=w^vdhInV=SvdUyt@dCwR1fi2r+p|N9&K-xFm*w=nTqn1N$$$wC7Nfn{R1YbB8~kdq3)X}?1NXI;FKz;Ary#Fm&}X7DI|!n3iY@Iw7?u%AlxmA1W|h*R6zk==Lshw6An$>*@PXCJ{cn$rYM6Pds9eh;P~GL?`4n+_=@{o|q}AfbqlgpBXX-l7aGNin*I7R*Q|D743f1_3>JG%J}iN<DLzpd26yv7!$Jg|<kW_^zdN_jooSMEANylM4OdOeCM?L>HNYNec_a1*z)yc9zT+io|Z1w8F~!g3Dad2iak2m3`Ma*d~}zOOZ+zKCQn71gd)AKdpQ;Rli7!uA#tM*u8_B@S~TxgQo&mwEhU(pSRHp28hM|0h9j3=iX5C>$%SD#VR@rg#iSr1G}cp+Hr<8i&2ODfo*TA1cVhHK&#!Rp*q^L<q1KJksKg(1@T}XWyqd+DC^qfziS#q#=V^+a<Nf_%4!m7tBG&YKbh@&uC2G*-25%D$PH~UO^#}g3;(96=iZLJ8z{K-t(n`9;nYlurPCZ+Q&A_a>M5CiQ8{H>CWE<z?GT3;k7Ubx+H&oXEJs-Vfl$3=b5})dbuVHZ*p@7*i|r~DakUbL$cCX{10@vM7L34XV!z9Bj;c2JXYmb4*4JLXRq+?>dj;##;nfvN2v4=ca~aRhH*mVb5daHy?XgK%o6yxzXapi-BQ8c9IfA&RLRG~m7e9)$oggSYr;uZEbwHxOj=)C)(<khKOFAjS1q{vfW2sG8dB+2=^YIKB&Ke!lQ&hu&5|aY`VQ_veK%&FbS);+~Ys7nY9p&c~M1kYm^L)WxXpc<IXO2GU@|sMAs_zh>lZcQD4p6mGYX{{gCxs#pz#<#QPpiNkuIF$P_|;Mal<MJD@T!JAqUqXRzvzO@h(gNfA6JyDnpix9LW-~fow(pR{;Nq%0hB3=2H1Bo$gZ`40|ZWw`BSt>0CA@AB`t{Jo>_i9GMG1S7aEaFbOjR(0<}059zaA`Wz)ca$aJ$Cxo{{!(KU`dMvn(9*LecsB4QF<Iwz;v{i+ttSiY)jRz)l7mQC8qE_yZLj^z2~qSC^u0f`iA0;`9f)I?70%xcmE_GeK9ACpP~1K54xI;sRIxKCta4PapVe(p^quU_nZ_x`oFP6~+ooGne7sE>)IavFmn0U&EJt2O01huFmIh$U*P8&d-yRe8U!WbMzlDyXRmXRSD*R$N%S12{FVYLH6>0~sd-!$fEn0(YG-qn#H5ASv<%ODt@*0`|Qu4r~*1L5P)-y(5Du2h|YJk~8(mOb+ZYj@EYP>}^o5Mi}83)O`!&Ix^>2#io7Ta736f1g46Mgkd64X~X{My-=}?isfFB{1Ug(WKC@|NvEMNlb6kk`1^F~`HlBZg(ZQk4*pRw#fWDQqC2yvUIr>(eu)#+GQlba=301jV9F~BLf~~b)I5q>gg_+iCJa=}$hfYJ8a8;UF(78`(B>DUpC+q0a!so0*4OM_sfCS(c;)OCo%C*(BURvKkqy;iN`iRoGql54yOG-8wHYnlILguW2r$%<b#DN?vt=|b`&w9+>aJeN4qJ6Ai%!zl<Wpw_H6CF8tMLOF_#NE9Hd%77_u&pToFS(WIBnb_s8IuR24$M6v62cFibT~S3OD&g{C4w~JB`q4+3RL!f!<|$Zf?a9{l6h@xc50V9e4rmINt7^mf4Q|Y`?5)=MGJ*E}-8jT-myq5k9A!6QX!AAhkb%Z29&~y%<7)V{p7IF@D3rM)(BSWSfBE?9i3j@BL*R6(3`KF;6OOcUB^fEyMLb!ZP#lp@d2)F*-y`>%>%!dTq96zNX`juIe&*_u5^42AQb#iwAY;*>e$2lMJoh$I8e(oj<J<eQ&EmrKYqC4x$E9{F*3ndRY>*iLj!h{2y4t(g^xo>=kG=jwqFKpn9`R9!iJn^(c_RBTLx3JUc@SX@kW(2-q*h6an%-UY}ZnT@A==rA*UP<cwZrEKy^?NM(Yg%@ioIgdy02$}og@*VSUcL3S`SuyM?{E1cFUI|GITi$AgDcnnQK3zUA}mv{smA8ZW$0weeDk|V3i+AQHAt>WQ~0OK>jp~38NxFbDUd@zPg&lSW;=d^5)`feT`t60E@<|kMeU}U%gRJLz7R+>+Y{%F3-7Su)zuH18sR+4*jBk;zYL#1jxitQTc<C@mPz&7<6kO^kKU2h8T$EZo68O6kfuV)}{bU~M;-Pfp*W(KbK@{_z#Pu<WM6VOD(f|}^*-y)$E;F}MlVwNN-#1%byt8K9~jHVUZY_!|cs*uMaxr&tGJTeVDZ1crpOC`2FC1l#GSO8RZ8qN*xow6dQpt7JoP3C@RWpJFRpis_0ynWJgL_t0U3W9;euJVae`33zX1wkX3({Tes-KIsf0JOpVG}y^b;fIDGr||;xEsUiO)i?s4U%#f9YTB$RTK10kV&WY%-^;3=s2``zY83Q?-8W?(bBi_F-H-0c*MtGOfGRPT?+_#Bs}O0l7Yb!?3ECu+x%q-DJG;PC0g<{L$qED671}qt6DTyPn6NAp%Ybem<%L-R3*d?|u%8_V%fd+Ixz(U*Z|(;U{YXw|>IGT{cLQidJsHr9(!MfdA%*P@umQ^ia~}pJCVP9Y{B`3Hg42RFckPs!)VOWS9?|E5aaJf-PW`0l+R)8$e%)lSJB&cx(L|}@PDczlU@*oDBO@LUIE?jRq$5;{$s_BMM6V-KH8|XdhF?A&_2ml1<lqSw!JU>6Ow$w89tQ6F6TP<Z^CDDN`wQK==`Dger7Ze#EkrFkF^Cr2G7^fZhV#d~k4$pYMO|LHKskw&E%G4*jJ66@-S+w)RCP1PsL;YxU6d+BmSQ6SqnPyeLmUha(=-{OAud~K3koJ00Q(72U`bO^<3}2=Ch)L!;iY^0d~;Y$fMd~R_^KazBQr535Y$3j%{w=sMd?9T<<85eJZli0VenO5R{qs-{3FHM{fNGk&dF5QFARB>PI4GYI>g1f=8zm?B2CQFexV7d3mDJ5{jP3bP$dV}u+ua?i)xtTjbjE9tINqmq5QT^r_$(|bz`A97^&AT&?7YLQ${2=mn!Z28&wi;fI2b{zAmEqhx|Z2I<U7i!UrRAzv+gAy^8U>a0qaE31zB)TgzgrsxB*_XEj9AQxKAuH4Gi+md~}jnoCIq1|$VdXVyI(M+agTAA9<uk8fUpLeb|haRth^aSGj7*K#nONxDeR+PYHwvduH==uYwkjFl5pI1`$f)Rn)nzCR}yO^WKK36OzL6oQ(E`<{fqxrXct<bS>{MobCN3@+K1c=3^6$fI^1iz!XAkSL7t^u-e|UP#M2T6qq40D&x6Xtis5y<-?2!SHw$c6A)LTM9eIHqZm(tH~<vfxNx<@bvC%z%gwN9X{TjJ+-!HRGhH#qzriX^wPT5=150~bUA68ky8>#Vi~BqxV3>>y%zlIWX$zX<(@iOf6z8z+LZ$HLM4g<+`Ml{HEINB5D&DWo9$J4o`8(mj<shfh63)s^lag__13f0V$H0IHdbRgLExt@tf8E$8ihYW<O5YVR`568TRib*bSUxQ!7&IVaR&23m7ZIVDc+N%62?F9I#^%8-IngkEfbCk#h6gKTW%T9E7Z^+>US(J>FUhqx><a;7E}|G?MF)rHo`~h8%ayW%{`%Cp$)}usO^xh3Ty~PLR5>@(>hIc>uj5bD1fT~Gn8Oi8doQdkv$8HumeXHE|P3SE8)N7bHn@<#>H69<^}8xUyL9v7DO!u6+I2qTB_6$^i!{Ip&dl0?a5fMnd$`FnVG=??%U-SrZ?yhg*|A9u+B?p8rQ5VTGr;1w)3gMnr%COBNVH_V0Qwex%nw8lx;>Uw&H6G_<H~!drGZl+mU1iAW1(RMwGJ)%GqXz0hf@u$;#bkTLqi3GL>QQ^P21$nUVB;)MRmIFixYf*`RI`@g+n#tfi@`te18;(7m8D1ndV6H_*P$#7?BrGgWq~8h$McmW;=&4*Zog5j)VxtU)nFjnPWPw^8%fcL3L_j*^9C^yausjm!#DL9W8s^>q%TX0j_%@ItCg@7VTL!$jS_CcR2+Tpc0h=Hlq)6p=1%aAe!`95QaC6+l;~$_&?U2dgUd>Xa&}>mV&h$UD~Fp<MZn-E1y|j`78efC4mR<`QUA5}RYd-UI3mcZOmSJh6TXyAQI4RO#B_R`!b0_$-nf)AmHOWytIs<nxfP71CaZQTp8~y+W$@yZ3K@`0(<5c>MCiarlR~FJ7AZG!6wl0yv5WW?4v{qTdj;&|L9D(1(bl=9C~n&GD8oi9CL1@mkBl)Pu^?;*5=wI5@}I-3XbgQIE+S+u5Nr^zeSn(K#4eMx8T*;nQz$W6t9%pR#i8n}gR&f(uM6#YMRk=%TCMK`o@)QJsb&g6rjyLp49e3QX<<SQ5rD4+z$P1`z5B<9ZwX%rU$_F^<3&nYu&+%Gs83Z9<#9gzD)VX@wG-@B))c0>fST1!v?^XPTUaRWpm;FoGGEP6jq^7iYK%v?qTJN)<(b?N5Au1w?54?#$)(`}mrF;hjHsBDWfKsFYBc<4E<O=tGs(l6D|rn3dKsm;~k=ANxjKOCAYEAP^}A1wn<&auzCe7+`dWYV$&nJH9KbSiwX+&48p6JvN4E)}r-&HCI$FL_yT3ldA>-sq#G0UhSXkpJJ=3qNXo=)u3ZV3PQ?KU=cGq=5)=I;lb&N?Df;WSc4QnX+uy=&;vORB;MFQK6aNG94ETmE^}P>YaVp9TVB88{J-{XFV_)TMF2JUAc}1ey%P@!?iedT08lh`W(yTO)bGKYeMz1Ay}s5Vsl<fQ6p%ZfuA=nZ7eT5fTxu`jxQ*GT(FI;8$3zF0QI6@in(<q;nN%&rj#n929N<n{aiSzB&9h+Q$N8WCEnRx^?e(vJ`iIgxU;g<&z}~U?^*{gh+WYT+{GWgS^-upY^U`1c^iK)Yfj84X|8oLYG=Kfi|0+jz)=}-RM|9R`l5C-<FhX=z4!d5sj!GQ|#t$k4-33+2I8;Vs%Bp~%Z)lmun%9o#85DGo1dURP>i%2%d6YmJ#rzm5dShVma$(T`;OUr3zA7qd0|vSr1<qO;Q{&cbK<#XDwbN6wO|_Oo(h(6!a^_AMD{16dTP0QWuCuKN$;!Ot7VZk^8Wv|Ej}#VSuA_s;`^`(EwMAp&Z)Gx3t5ma?JVcXLl_GfHnOWiLcjhVYL?H?zuqn6#XQW2aDZUjfeXq~a_A;I_(G=SaHX3acY5qezeL?SjfT8quNNMCwu_n3Eq))#q>l)8f94C&9U^VZ{h(tEJmgtUkl<CE#+CC<h1M{KLgxHM9t6Vhz-dP~Th8{knRznCe0Jn^mrH%T^ITICiu`VMcw@Oyx8b_T^T!NemVF!>67xc!id03PY2tAt=+lNezkhDE$a;`h)>Yk)xiz-(lTsuu9<%5;brg58;b1?9H`IfShTdlK8YS4pWwh}R>!CGhYO*mW6!PCKF+BaDB7~+YQlysuxAu2fK2mF!Dtz$*a%@iO`DGjDW`0gaR>Kv=i+EYU|@71pCFfdvbzHJ0C+n{2AR8}sAPy9RLob_J**f{6U7=>#u`e(|$f<fVW##-su@!~B3-q8*7t=rKiJ*#HRGsBjBOcjgA>iQgPkAZq9I8=j-j$w9Qi{b&vN4OZo;2OgSaE}|X`|iC{k8>yP08a*Lb_zhRzFP3qqtGr0a#7^sq`!}dH*dyPci;HeGvv*~H_*|{;1=Vp>-Nq3ivmvxyOdG}e|20FySa5<1G_UNg<5di$c8~ZKPh8smzTs$I`KeR6<gI2BM-%chKq#FrtqJUtb&bIf(hH;b~<iN%TlW-+L^15Ps%!=W$^z$DF=*<c<o67R8YRtDETqA^(vRJq=s*|Yjr9>t%_cow$lhqy7QK;m87cP`ti<Jbs?3Ng!ggQT6=TrGo9k#`o^UM%mf8KNlNS|U((@QG#=g^0UblxWcR?Amcz4r6vgn-ST_??|CnEgY)BG^ZTZAxU9duPRdhJMs1H@Ooh~|KGr8qlHEqA+%%sBkKrznWn1kRiF#Ih8re)1Y&*`%inU?8PFkc0PiwATMOLSNWMXO>$<>C^2Z6sHSbD-<S@G5wZC_3jNi7(;dkRF&pj=h6hnsYVzMno{ozR5>v3PvlZ(*)WMQPFDxXU_!Dy`EgMFwsOoYRaae(VgYB*>ioo*l53C&fu`feE<Ay_~N^_M=I5xR;CzI#Yv>oZU|PYYT{GoD)>j=e)l{)`u6$X`?f{C)q;+O9i4t!Ij?^I?b|nh7wS&?&R*4q1`!WMp->s9&|i)m%8J{qHuNhDri5>1v2)BjqDCOnP@Zx4%2N)X&4R6kX9|2BX&T`dY+i$@^GatF_b^9&VJ1as#KXdfPz%P0yq4%Iig2ep3t=;ZFb3$WamIL6ZFY_ZYRn&UdTToHOn8>hVuj<i89$Dvl65dAeX1@9>+$Vy*;3zyR4A6<DFwaXbh2Wx`ZSHkt9;Lj0@k27D$j=pqy15ljrL)N0hdA1aKHTRmy{;5DltR#rS&vVAU0%p*aX6_4jZt{7j=+nQ9J@b0tO#}6%F{^CjZiU3D5Y}f^arn<46vq7Cawl5Lf3@wz-&t)usClXn#Z{2IEU(3+R<Du3-Oh9-+U)uBXZlns67%6&S*y?jv27imK`U5kH)eW0gZ)g^CUv(7*!<0*+Nx6&)IgR|R#1&oHROY={L$GkUBy+CVfQ<oiqVtU*2Yt6_!F+v+qC*SMxj)7LGmLA#5@Wu$7;ePP|O{uLreE<E*#yV22$*RKNxQOjeXq*=}^-2}h`60x`egZCR(Q?sll;h1aB<O%X#Wkpv;1cua@Qle33k21nk=riyu(ye{^=H3WCmoo!igj^J(TQO5x;{`SGorw+Mwb=KmR$F-Dd&P^`nf(Wwt?M~I^>`e6x?67xGA&4MDA_@XF~k}m!Kp7o@g)280IwK!h)ov~F-D%Oc$>@v@|*2DE}LxUSpz#*=sR;^6N87hU8(Txx%1KHntqk*D9<RJ1SLZRk$t|x&`Ipam2se&FxCL<4AWXhRQO;JY_xQ(@?Mhd7Er;Z1MkJl-@SVNCj938+c(EA-@KT>HzZPwk<V3VA|U+rQTY7z>xmCXe9yKsqTT}maeNR4kmG=Kh(bpNx*}F3u%P<VBSzH)JdrOv14>cO@$!vP^WCd=FR2{V7_<Dt@r$?L9q*o)9>qej%AjQoR%GIfv4d13QrFeT&vpk3Se+R79@I14H`h;v7xLlF;?X=tCIaKa&k}((5Cm7s)Zs^dbGQ^d2$kGK2rrb75a>^-H4)v%jRhA=FqVho2o&a7Z06U_J;3!9e?t@F?wi5d0m!n6ib#QFeNF9o=a|6h3;NOcv__I9Ii3*9^OYq48OQMFpV5W{7q@CmgT=Qd+o&BWPR1(r7>afDoN(h)H&!%rYi@@VVt{xxfIdCq4+Q>c>$bS>R~cG&QRnWsZ!e9_)V@g`?YCr90fB~`K;)SRUr&?+W6u2}DIfn>JPdyP@I@b>Yn<Bj(v~Wvs$P+>x-dz%D0I)((}?11g98<R0m>lQBmDNPDnef(9q@B|a`t8dYj@0k*Y0|g(fKS?S&<sjDsC&MoMUW-B15YeT=6Vrz(wt_b!;|%q}@i*Z^L1l4Y_oPIw2&CNa<eVLxFlas7x4jZOEZxol^XYdT3}*A}Oid^rs!NmF8d0hOV3u9o!4gZ^&W^N^6J2($baX>uBnL<W=)`HK{5euWeG*ACle0mfHV}I;Q?2(I<hjRaLqF8{^ll#N2A%|9xm)P~c0me6mi#eE~OR<sxn69iHIT-uBD0yVx(9?sxIDYcG7KhhD)R(U&V{`Wx0C)T0_1`~N2u#E~Myx9Sa(HrE?LXk#dcd;mKUukoQU37?s{^t-FVH-nLchx7PiSS;7WErrc>w^3gkh9=YP8N>vf4L8Z=t~SaiJ;cOXlY_%2`-89c_p2zQY8fjUZ-az&hib`j+toj)f-zf$Eru93bpGhjzZ*1gumV{BG`p&XW)V~qgeni^^BL@~w;I`qse%`vBGC33`f>djO8I-vQV}B^X3-{^CgfK0&oYqxK+HpoqbZ!i4K4RE@F9@j<?C;i?0@2aco`l&Kl=7%`0n}fw>qi-$)4SqxJ5@ot0Aj`RX#i?#N$_gcnMO3;$9%6`TXHHMb;CcXiDA#J#A2u(SA-Aftk2T_HC0mHYgK^=a=w(JYomv_mIR7hD>wx{^j%Im*JbY;SaCgym<S=ZG#jhniP~_Q=%Z%ARN0b;K@w*??4u3!haocEgv7J(uhfr7|}STy!Lk3=H0B;5WR%m-dHHsHLIJ-!lH!kP%$w>1i$B+wZ*cp7<Y%SvG*zi`zkb2pAW2qQ^e6V9<IR&Jbe?fc&pcVG;mhO5G>M`qci1h?KTCa{alO=O&>l_!eI05A3Wf}gSKzpp;D)1Eg%r~^C~%8mh}_vp%9ujnt!h%+{9K+mS*{iULhAWaq4I`$GB|m14A=I<^2+x*EfS4R7?>B?gLmKo8};$kyjL&sv`~t{1s^eG}QBcwx^1pVa&|eJUNnxp6yYQ?+T+(-^4%;UjygwEY|5J>bs`%M0xR$fJWUil9_(qZ!DzVaaBSdmhjx)DcsSQ{_*4JIdW&=ptR7aQ`X$+gIAx#YoEn^2HMzy9Yb}r`RwD~iBuHMo}wiKZ?t+Y;Z`Y<wL=a_N{NN?DZ<8uN=@9(h(l&2W^Np}Xl$m&`6wCL4g;JVV2LBK&Y>%9O*2moaGmF{uRu;&K`4E;!VidfBBg3xroMOg)KkJjK@q$K%Hyb)6WTN{<10*^jz38rs8ie2a*XH%HdgAu=A|r&*>5DgY0n;LBrPZ!H(B+azq)E<XH9Lr$l86=-tZ`Q-n5sr_VqjT>VW$!XGHA^`Q(liOtvY=qEYLqMyySl>P{{y!DTo9C(Pv@BAbI7#`d={cj6DNiS1%_+iB;#V+JuaKIhz#hv6Qd&0PXjXOutFVUcxx)cB5%8CUlc?-jP+N>TSPgIWyJ8TN07!#mKRKZMr!+#|<o@V7qbgO1gGREmAUXcG0|6S}EaI6up%V?Z21Yz9w6RjK=nfo{vC@p*ele-LmQr9%)A_05N_Z%ftIte7xbGy>Iaq6soKe$@%@FA@}Z53||*(_r5#>4l?hBkzTyj<HB8^_N=`ai?`?GMoPez6po-'}

EMBEDDED_SHA256 = {'audit_snapshot.py': '75a6118be7c78e17a4fbc1a2c2fd50ddbf4616bdfc5750498a7afda7c808b558',
 'corpus_runner_snapshot.py': '147eed707d4f425d88d0579befaa1c2e43a856a9a2f767ba4785f083249a4c98',
 'crawler_snapshot.py': '2a074ad59fcc4a90d7137c914af51fd733905d5715044f91f4dae4b68b6f5022',
 'diagnostics_snapshot.py': '935ce5d35a3ac143cca46351a9bc408c56aceac930fe4e40b72bf224adefb0af',
 'export_snapshot.py': 'ac461b827b046ed7ea8fc4488a85d62392e4a95920ca9be7e8d714d3737a4023',
 'postprocess_snapshot.py': '377d747c79f82e52d4fd9f16e36512af540d985536aa0a70b86855eff76842b8',
 'priority_probe_snapshot.py': '608a80b2281daf534180ade41eaed2e458e6e949e51402eecdb4ace9ed5e66e6'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
POSTPROCESS_FILES = {'audit_snapshot.py', 'export_snapshot.py', 'diagnostics_snapshot.py', 'postprocess_snapshot.py'}
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if name in POSTPROCESS_FILES or (not target.exists() and not POSTPROCESS_ONLY):
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
import importlib.util
postprocess_spec = importlib.util.spec_from_file_location('notebook_postprocess', RUN_DIR / 'postprocess_snapshot.py')
POSTPROCESS = importlib.util.module_from_spec(postprocess_spec)
postprocess_spec.loader.exec_module(POSTPROCESS)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

def save_checkpoint_archive(failed_step=None, error=None):
    archive = POSTPROCESS.package_run(
        RUN_DIR, WORKDIR, id_from=ID_FROM, id_to=ID_TO, dataset_revision=DATASET_REVISION,
        source_sha256=SOURCE_SHA256, packages=versions, refresh_diagnostics=refresh_diagnostics,
        failed_step=failed_step, error=error)
    print('ZIP checkpoint đầy đủ:', archive, flush=True)
    print('ZIP chẩn đoán:', diagnostics_archive, flush=True)
    return archive

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException as exc:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        try:
            save_checkpoint_archive(failed_step=name, error=f'{type(exc).__name__}: {exc}')
        except Exception as rescue_error:
            print(f'Đóng gói cứu dữ liệu lỗi: {rescue_error}; checkpoint vẫn ở {RUN_DIR}', flush=True)
            with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
                stream.write('Đóng gói cứu dữ liệu: ' + traceback.format_exc() + '\n')
        raise
    else:
        refresh_diagnostics()
with diagnostic_step('snapshot_validation'):
    if (RUN_DIR / 'crawl_config.json').exists():
        saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
        if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
            raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
    if not requirements_path.exists():
        requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if not POSTPROCESS_ONLY and Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

Tiến độ in mỗi 25 URL hoàn tất hoặc khi hết lượt. Chế độ robots theo `RESPECT_ROBOTS`,
giới hạn một request cùng registered domain, có retry và checkpoint theo ID.
`WORKERS = 4` không có nghĩa bốn request đồng thời đến cùng website.
ID đã có checkpoint, kể cả thất bại, sẽ được bỏ qua khi resume.
`MAX_NEW` giới hạn URL Requests mới; Selenium tiếp tục các URL lỗi, text rỗng hoặc
QC lỗi đã có checkpoint nhưng chưa thử browser. Chrome được kiểm tra trước lượt Requests;
nếu không khởi động được, ô crawl dừng và ghi lỗi riêng, giữ checkpoint hiện có.
Kết quả cuối ưu tiên engine có QC tốt hơn, giữ Requests khi hai kết quả tương đương.
Parser 120ask giữ câu hỏi/câu trả lời; cnkang giữ thân bài, bỏ menu và bài liên quan.
`records/` là kết quả cuối dùng để audit/export; `requests_records/` và
`browser_records/` giữ bằng chứng của từng engine. Chỉ nội dung qua QC mới được xuất corpus.
ZIP `<run>_diagnostics.zip` được cập nhật sau mỗi log tiến độ và khi lệnh kết thúc,
kể cả ngắt ô crawl. File có một số mẫu record/raw để debug, không nén lại toàn bộ raw.

Mặc định `BROWSER_TEST_MODE = 'failed_only'`: sau khi Requests hết lượt retry,
mọi lỗi HTTP/mạng/timeout/challenge, lỗi trích xuất, text rỗng và QC `review`/`rejected`
đều tự được đưa sang Selenium, kể cả PDF cần kiểm tra lại. Log `selenium_fallback_scheduled`
ghi số URL được chuyển và lý do. `records/` giữ `selenium_fallback_reason` cùng lỗi Requests gốc;
nội dung Selenium có QC tốt hơn được chọn và xuất vào corpus nếu đạt `accepted`.
Chrome/driver không khởi động được sẽ dừng sớm và lưu lỗi riêng. Selenium vẫn có thể thất bại.
`off` là lựa chọn chủ động tắt fallback; robots bị chặn khi `RESPECT_ROBOTS = True`
và URL không hợp lệ/địa chỉ không được phép vẫn không được chuyển sang browser.
Sau cập nhật snapshot, dùng `RUN_TAG` mới; có thể đặt `RECOVERY_RUN_PATH` để phục hồi run cũ.


In [ ]:
with diagnostic_step('crawl'):
    if POSTPROCESS_ONLY:
        print('POSTPROCESS_ONLY: bỏ qua Requests/Selenium; giữ nguyên cấu hình và snapshot crawl.')
    else:
        POSTPROCESS.invalidate(RUN_DIR)
        corpus_settings = {
            'user_agent': USER_AGENT, 'delay': DELAY, 'attempts': ATTEMPTS,
            'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
            'respect_robots': RESPECT_ROBOTS,
            'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                               'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                               'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                               'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                               'robots_failure_ttl': ROBOTS_FAILURE_TTL},
            'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
            'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
            'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
            'browser_attempts': BROWSER_ATTEMPTS,
        }
        settings_path = RUN_DIR / 'corpus_settings.json'
        settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
        arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
        if '--log-every' in CORPUS_RUNNER.read_text(encoding='utf-8'):
            arguments += ['--log-every', LOG_EVERY]
        if RECOVERY_RUN_PATH:
            arguments += ['--recovery-source', RECOVERY_RUN_PATH]
        if MAX_NEW is not None:
            arguments += ['--max-new', MAX_NEW]
        print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
        run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Audit chỉ chạy khi đủ checkpoint; raw rỗng hoặc HTML không parse được được ghi cờ chẩn đoán.
Sai hash, ID/URL hoặc thiếu raw của bản ghi extracted vẫn làm bước này thất bại.
Corpus chỉ xuất sau audit đạt. Thành công được ghi riêng trong postprocess_status.json;
run hết pending chưa đủ điều kiện thành công/upload. Nếu lỗi, notebook tạo ZIP checkpoint
đầy đủ và diagnostics trước khi báo lại lỗi gốc. Accepted là QC tự động, vẫn cần review nội dung.


In [ ]:
with diagnostic_step('report_audit_export'):
    summary = POSTPROCESS.run_postprocess(RUN_DIR, SOURCE_PATH, run_script)
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending']:
        print('Run chưa hoàn tất; chưa audit/export hoặc đánh dấu thành công. Chạy tiếp ô crawl.')
    elif summary['accepted'] > 0:
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Audit đạt; không có tài liệu accepted để xuất corpus. Xem QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
with diagnostic_step('diagnostics_archive'):
    from IPython.display import FileLink
    import zipfile

    refresh_diagnostics(strict=True)
    with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
        diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
    display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
    print('ZIP chẩn đoán trong Output:', diagnostics_archive)
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP được tạo cả khi xử lý thất bại; manifest ghi complete = False cho run chưa qua audit/export.
ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
with diagnostic_step('checkpoint_archive'):
    from IPython.display import FileLink

    archive = save_checkpoint_archive()
    notebook_manifest = json.loads((RUN_DIR / 'notebook_manifest.json').read_text(encoding='utf-8'))
    print('Run đã qua audit/export:', notebook_manifest['complete'])
    display(FileLink(str(Path(archive).relative_to(WORKDIR))))
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
    if notebook_manifest['complete'] and notebook_manifest['accepted'] > 0:
        display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
        display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))


## 10. Tự upload output lên Hugging Face Dataset

Repo đích: **AncientWings/AncientWings_R2AI_STAGE3_ds**. Bật Internet và gắn Kaggle Secret
**HF_TOKEN** có quyền ghi vào repo. Run All sẽ tự chạy ô này sau khi xuất corpus/đóng ZIP.
Không cần đăng nhập qua widget hoặc tải output về máy.

Mỗi run được phân biệt bằng `RUN_DIR.name`: mặc định `ids_<ID_FROM>_<ID_TO>_synced`.
Recovery có tên riêng; nếu crawl lại cùng khoảng ID với code/cấu hình khác, đặt **RUN_TAG mới**
ở mục 1. Chạy lại ô upload cùng run sẽ cập nhật đúng các đường dẫn của run đó.

```text
corpus/<run>.parquet                       # Đầu vào RAG: chỉ tài liệu accepted
corpus_jsonl/<run>.jsonl                   # Cùng corpus, định dạng JSONL
archives/<run>.zip                         # Full raw/records/checkpoint để resume
diagnostics/<run>_diagnostics.zip           # Gói chẩn đoán
manifests/<run>/notebook_manifest.json      # Khoảng ID và số lượng bản ghi
manifests/<run>/upload_manifest.json        # Danh sách file, kích thước và SHA-256
```

Mặc định upload tất cả các mục trên; có thể tắt JSONL hoặc từng ZIP trong ô cấu hình dưới.
Đặt `HF_UPLOAD_ENABLED = False` nếu chỉ muốn giữ output Kaggle.
Run còn pending (ví dụ thử `MAX_NEW = 10`) sẽ **chưa upload**; chạy tiếp mục 6 rồi chạy lại mục 7–10.
Run hoàn tất nhưng không có accepted vẫn upload manifest/ZIP; không xuất corpus cũ còn sót lại.
Upload đọc file từ đĩa, không nạp cả ZIP vào RAM. Nếu mạng lỗi, ô thử lại tối đa 3 lần;
có thể chạy lại riêng mục 10 mà không crawl lại. Kết quả commit được lưu tại
`hf_upload_result.json` trong run. Repo đích phải tồn tại; token/repo lỗi sẽ được báo ở ô này.

Tài liệu: [Hugging Face upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
và [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).


In [ ]:
# ===== UPLOAD OUTPUT CỦA RUN NÀY LÊN HUGGING FACE =====
HF_UPLOAD_ENABLED = True
HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_SECRET_NAME = 'HF_TOKEN'  # Tạo trong Kaggle: Add-ons > Secrets; không dán token vào code.
HF_UPLOAD_JSONL = True
HF_UPLOAD_ARCHIVE = True
HF_UPLOAD_DIAGNOSTICS = True
HF_UPLOAD_ATTEMPTS = 3


def upload_current_run_to_hf():
    import hashlib
    import importlib.metadata
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import re
    import subprocess
    import sys
    import time
    import traceback
    from urllib.parse import quote

    if not HF_UPLOAD_ENABLED:
        print('Đã tắt upload Hugging Face; output vẫn được lưu trong Kaggle Output.')
        return {'status': 'disabled'}

    run_dir = Path(RUN_DIR)
    manifest_path = run_dir / 'notebook_manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Chạy mục 7–9 để xuất corpus và đóng ZIP trước khi upload.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('id_from_inclusive'), manifest.get('id_to_inclusive')) != (ID_FROM, ID_TO):
        raise RuntimeError('Manifest không khớp khoảng ID hiện tại; chạy lại mục 7–9.')
    if manifest.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Manifest không khớp revision nguồn; chạy lại mục 7–9.')
    if manifest.get('pending', 1) != 0:
        print(f"Chưa upload: run còn {manifest.get('pending', '?')} URL pending. "
              'Chạy tiếp mục 6, rồi chạy lại mục 7–10.')
        return {'status': 'incomplete', 'run_name': run_dir.name}
    # Completion requires fresh audit/export evidence, even if a legacy manifest says complete.
    import importlib.util
    helper_path = run_dir / 'postprocess_snapshot.py'
    if not helper_path.is_file():
        raise RuntimeError('Thiếu bằng chứng audit/export mới; chạy lại mục 7–9 với POSTPROCESS_ONLY = True.')
    helper_spec = importlib.util.spec_from_file_location('hf_postprocess_gate', helper_path)
    helper = importlib.util.module_from_spec(helper_spec)
    helper_spec.loader.exec_module(helper)
    helper.require_ready(run_dir, manifest)
    if type(HF_UPLOAD_ATTEMPTS) is not int or HF_UPLOAD_ATTEMPTS < 1:
        raise ValueError('HF_UPLOAD_ATTEMPTS phải là số nguyên dương.')

    run_name = run_dir.name
    files = [(manifest_path, f'manifests/{run_name}/notebook_manifest.json')]
    if manifest.get('accepted', 0) > 0:
        files.append((run_dir / 'corpus.parquet', f'corpus/{run_name}.parquet'))
        if HF_UPLOAD_JSONL:
            files.append((run_dir / 'corpus.jsonl', f'corpus_jsonl/{run_name}.jsonl'))
    else:
        print('Run không có tài liệu accepted; chỉ upload manifest và các ZIP đã chọn.')
    if HF_UPLOAD_ARCHIVE:
        files.append((Path(WORKDIR) / f'{run_name}.zip', f'archives/{run_name}.zip'))
    if HF_UPLOAD_DIAGNOSTICS:
        files.append((Path(WORKDIR) / f'{run_name}_diagnostics.zip',
                      f'diagnostics/{run_name}_diagnostics.zip'))
    for local_path, _ in files:
        if not local_path.is_file() or local_path.stat().st_size == 0:
            raise RuntimeError(f'Thiếu output hoặc file rỗng: {local_path.name}. Chạy lại mục 7–9.')

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Tạo và bật Kaggle Secret {HF_SECRET_NAME!r} cho notebook này; '
                           'token cần quyền ghi vào dataset đích.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME!r} đang rỗng.')
    token = token.strip()

    try:
        from huggingface_hub import HfApi, CommitOperationAdd
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'huggingface_hub>=0.34,<3'], check=True)
        from huggingface_hub import HfApi, CommitOperationAdd

    def report_upload_error(error, stage, attempt=None):
        # Giữ thông tin chẩn đoán, nhưng che token/header xác thực và query của URL ký.
        def redact(value):
            value = str(value).replace(token, '[REDACTED_TOKEN]')
            value = re.sub(r'\bhf_[A-Za-z0-9]{8,}\b', '[REDACTED_TOKEN]', value)
            value = re.sub(r'(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+',
                           'Bearer [REDACTED_TOKEN]', value)
            value = re.sub(r'(https?://)[^/\s@]+@', r'\1[REDACTED_AUTH]@', value)
            value = re.sub(r'''(https?://[^\s?<>"'\)\]]+)\?[^\s<>"'\)\]]+''',
                           r'\1?[REDACTED_QUERY]', value)
            value = re.sub(r'''(?i)(\bauthorization\b["']?\s*[:=]\s*["']?)'''
                           r'''(?:(?:Bearer|Basic)\s+)?[^\s"',;}]+''',
                           r'\1[REDACTED_AUTH]', value)
            value = re.sub(r'''(?i)(\b(?:access[_-]?token|token|api[_-]?key)\b'''
                           r'''["']?\s*[:=]\s*["']?)[^\s"',;}&]+''',
                           r'\1[REDACTED_TOKEN]', value)
            return value

        chain, seen, current, status = [], set(), error, None
        while current is not None and id(current) not in seen and len(chain) < 8:
            seen.add(id(current))
            message = redact(str(current))
            chain.append(f'{type(current).__name__}: {message}')
            code = getattr(getattr(current, 'response', None), 'status_code', None)
            if status is None and isinstance(code, int) and 100 <= code <= 599:
                status = code
            if status is None:
                # hf_xet có thể bọc HTTP error trong RuntimeError mà không có response.
                match = re.search(r'\b(?:HTTP(?:\s+(?:status|error|code|client|server)){0,4}'
                                  r'|status(?:\s+code)?)\s*[:=]?\s*\(?([1-5]\d{2})\b',
                                  message, re.IGNORECASE)
                if match:
                    status = int(match.group(1))
            current = current.__cause__ if current.__cause__ is not None else current.__context__
        versions = {}
        for package in ('huggingface-hub', 'hf-xet'):
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = 'not installed / unavailable'
        detail = '\nCaused by: '.join(chain)
        heading = (f'Hugging Face error | stage={stage} | '
                   f'attempt={attempt or "-"}/{HF_UPLOAD_ATTEMPTS} | HTTP {status or "?"}')
        print(heading)
        print(detail)
        print('SDK:', json.dumps(versions, ensure_ascii=False))
        diagnostic = {
            'time_utc': datetime.now(timezone.utc).isoformat(), 'stage': stage,
            'attempt': attempt, 'http_status': status, 'repo_id': HF_REPO_ID,
            'run_name': run_name, 'sdk_versions': versions, 'error_chain': chain,
        }
        log_path = run_dir / 'logs' / 'hf_upload_error.log'
        try:
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open('a', encoding='utf-8') as log:
                log.write(redact(json.dumps(diagnostic, ensure_ascii=False)) + '\n')
                log.write(redact(''.join(traceback.format_exception(
                    type(error), error, error.__traceback__))) + '\n')
            print('Log lỗi upload:', log_path)
        except OSError:
            print('Không ghi được file log; thông báo lỗi đã in ở trên.')
        return status, detail

    api = HfApi(token=token)  # Không login(), không lưu token vào file hay biến môi trường.
    try:
        api.repo_info(repo_id=HF_REPO_ID, repo_type='dataset')
    except Exception as error:
        status, detail = report_upload_error(error, 'repo_info')
        raise RuntimeError(f'Không truy cập được dataset {HF_REPO_ID} (HTTP {status or "?"}). '
                           f'\nLỗi gốc: {detail}\n'
                           'Kiểm tra Internet, repo đích và quyền của HF_TOKEN.') from None

    file_records = []
    for local_path, remote_path in files:
        digest = hashlib.sha256()
        with local_path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        file_records.append({'path_in_repo': remote_path, 'bytes': local_path.stat().st_size,
                             'sha256': digest.hexdigest()})
    upload_manifest = {
        'format_version': 1,
        'source_notebook': 'crawl_corpus_kaggle_hf_upload.ipynb',
        'repo_id': HF_REPO_ID, 'run_name': run_name,
        'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
        'dataset_revision': manifest['dataset_revision'],
        'source_sha256': manifest['source_sha256'],
        'complete': True, 'corpus_policy': 'accepted_only',
        'counts': {key: manifest[key] for key in ('selected', 'completed', 'pending', 'accepted')},
        'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
        'files': file_records,
    }
    upload_manifest_path = run_dir / 'hf_upload_manifest.json'
    upload_manifest_path.write_text(json.dumps(upload_manifest, ensure_ascii=False, indent=2) + '\n',
                                    encoding='utf-8')
    files.append((upload_manifest_path, f'manifests/{run_name}/upload_manifest.json'))
    print(f'Dataset đích: {HF_REPO_ID} | Run: {run_name} | Accepted: {manifest["accepted"]}')
    for local_path, remote_path in files:
        print(f'  {remote_path} ({local_path.stat().st_size / 1024**2:.2f} MiB)')

    for attempt in range(1, HF_UPLOAD_ATTEMPTS + 1):
        try:
            # Một commit cho corpus, manifest và ZIP của cùng run; chỉ thêm/cập nhật các path này.
            # Tạo lại operations khi retry vì SDK có thể thay đổi chúng trong quá trình upload.
            commit = api.create_commit(
                repo_id=HF_REPO_ID, repo_type='dataset',
                operations=[CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
                            for local, remote in files],
                commit_message=f'Upload crawl {run_name} ({manifest["accepted"]} accepted)',
            )
            break
        except Exception as error:
            status, detail = report_upload_error(error, 'create_commit', attempt)
            retryable = status is None or status >= 500 or status in (408, 409, 429)
            if not retryable or attempt == HF_UPLOAD_ATTEMPTS:
                raise RuntimeError(f'Upload chưa thành công (HTTP {status or "?"}, '
                                   f'{type(error).__name__}).\nLỗi gốc: {detail}\n'
                                   'Xem logs/hf_upload_error.log để biết chi tiết; '
                                   'output local vẫn còn, có thể chạy lại riêng mục 10.') from None
            print(f'Upload thất bại (lượt {attempt}/{HF_UPLOAD_ATTEMPTS}); đang thử lại...')
            time.sleep(min(2 ** attempt, 30))

    result = {
        'status': 'uploaded', 'repo_id': HF_REPO_ID, 'run_name': run_name,
        'commit_url': commit.commit_url, 'commit_oid': commit.oid,
        'files': [remote for _, remote in files],
    }
    (run_dir / 'hf_upload_result.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Upload thành công:', commit.commit_url)
    for _, remote in files:
        print(f'https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/{quote(remote, safe="/")}')
    return result


hf_upload_result = upload_current_run_to_hf()
